# 12 — Model Geographic Workload Shifting

## Research purpose

Estimate workload-shifting potential for six and nine candidate hours across the nine utilisation and flexibility cases, using linear programming as the primary method and deterministic greedy allocation for comparison. Compare daily and same-hour shifting for both model layers and each weather year.


## Imports

Load the libraries used for allocation and optimisation, and check the pinned SciPy version.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import warnings
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
import shutil
import time
import uuid

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import OptimizeWarning, linprog
from scipy.sparse import block_diag, coo_matrix, csr_matrix, vstack

if scipy.__version__ != "1.18.0":
    raise RuntimeError(
        f"Notebook 12 requires scipy==1.18.0; found {scipy.__version__}. "
        "Use the pinned thesis environment."
    )

## Project location and paths

Locate the project folder and define the hourly inputs, workload scenarios, allocation partitions and summary paths. Keep the six-hour and nine-hour results in separate folders and define shared paths for their comparison.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_FINAL = PROJECT_ROOT / "data" / "final"
SIX_HOUR_FINAL_DIR = DATA_FINAL / "candidate_hours_6h"
HOURLY_INPUT_PATH = DATA_INTERIM / "cluster_operator_hourly_load_and_flexibility.parquet"
UTILISATION_SCENARIOS_PATH = DATA_INTERIM / "utilisation_scenarios.csv"
FLEXIBILITY_SCENARIOS_PATH = DATA_INTERIM / "flexibility_scenarios.csv"
DAILY_PARQUET_OUTPUT_DIR = DATA_INTERIM / "operator_daily_spatiotemporal_shift_allocations_6h_by_scenario_parquet"
DAILY_OPERATOR_YEAR_PATH = SIX_HOUR_FINAL_DIR / "daily_spatiotemporal_shifting_operator_year_summary_by_scenario.csv"
DAILY_CENTRAL_DAILY_PATH = SIX_HOUR_FINAL_DIR / "geographic_workload_shifting_daily_summary.csv"
DAILY_CENTRAL_OPERATOR_YEAR_PATH = SIX_HOUR_FINAL_DIR / "geographic_workload_shifting_operator_year_summary.csv"
SAME_HOUR_PARQUET_OUTPUT_DIR = DATA_INTERIM / "operator_same_hour_geographic_shift_allocations_6h_by_scenario_parquet"
SAME_HOUR_OPERATOR_YEAR_PATH = SIX_HOUR_FINAL_DIR / "same_hour_geographic_shifting_operator_year_summary_by_scenario.csv"
SAME_HOUR_CENTRAL_DAILY_PATH = SIX_HOUR_FINAL_DIR / "same_hour_geographic_shifting_daily_summary.csv"
SAME_HOUR_CENTRAL_OPERATOR_YEAR_PATH = SIX_HOUR_FINAL_DIR / "same_hour_geographic_shifting_operator_year_summary.csv"
COMPARISON_OPERATOR_YEAR_PATH = SIX_HOUR_FINAL_DIR / "shift_mode_flexibility_scenario_comparison_operator_year_summary.csv"
COMPARISON_TOTAL_PATH = SIX_HOUR_FINAL_DIR / "shift_mode_flexibility_scenario_comparison_total_summary.csv"
CENTRAL_COMPARISON_OPERATOR_YEAR_PATH = SIX_HOUR_FINAL_DIR / "shift_mode_comparison_operator_year_summary.csv"
CENTRAL_COMPARISON_TOTAL_PATH = SIX_HOUR_FINAL_DIR / "shift_mode_comparison_total_summary.csv"
OPTIMISATION_SOLVER_STATUS_PATH = SIX_HOUR_FINAL_DIR / "workload_shifting_optimisation_solver_status.csv"
OPTIMALITY_GAP_OPERATOR_YEAR_PATH = SIX_HOUR_FINAL_DIR / "allocation_method_optimality_gap_operator_year_summary.csv"
OPTIMALITY_GAP_TOTAL_PATH = SIX_HOUR_FINAL_DIR / "allocation_method_optimality_gap_total_summary.csv"
NINE_HOUR_FINAL_DIR = DATA_FINAL / "candidate_hours_9h"
DAILY_9H_PARQUET_OUTPUT_DIR = DATA_INTERIM / "operator_daily_spatiotemporal_shift_allocations_9h_by_scenario_parquet"
SAME_HOUR_9H_PARQUET_OUTPUT_DIR = DATA_INTERIM / "operator_same_hour_geographic_shift_allocations_9h_by_scenario_parquet"
CANDIDATE_OPERATOR_YEAR_PATH = DATA_FINAL / "candidate_hours_comparison_operator_year_summary.csv"
CANDIDATE_TOTAL_PATH = DATA_FINAL / "candidate_hours_comparison_total_summary.csv"
CANDIDATE_UPLIFT_OPERATOR_YEAR_PATH = DATA_FINAL / "candidate_hours_uplift_operator_year_summary.csv"
CANDIDATE_UPLIFT_TOTAL_PATH = DATA_FINAL / "candidate_hours_uplift_total_summary.csv"

## Allocation settings

Define the six-hour and nine-hour alternatives, allocation methods, solver settings and numerical tolerances. Specify the scenario labels and flow fields carried through the results.


In [3]:
SIX_HOUR_CANDIDATE_HOURS = 6
NINE_HOUR_CANDIDATE_HOURS = 9
MODEL_IMPLEMENTATION_ID = "local-clock-dual-allocator-v2-batched"
ALLOCATION_METHODS = ("greedy", "optimised_lp")
ALLOCATION_TOLERANCE_MWH = 1e-9
FEASIBILITY_TOLERANCE_MWH = 1e-6
OBJECTIVE_TOLERANCE = 2e-5
LEXICOGRAPHIC_ABSOLUTE_TOLERANCE = 1e-6
LEXICOGRAPHIC_RELATIVE_TOLERANCE = 1e-10
OPTIMISATION_BATCH_EDGE_LIMIT = 50_000
OPTIMISATION_MAX_WORKERS = min(4, os.cpu_count() or 1)
HIGHS_THREADS_PER_WORKER = 1
SOLVER_OPTIONS = {
    "primal_feasibility_tolerance": 1e-8,
    "dual_feasibility_tolerance": 1e-8,
    "ipm_optimality_tolerance": 1e-9,
    "threads": HIGHS_THREADS_PER_WORKER,
}
SCENARIO_METADATA_COLUMNS = [
    "utilisation_scenario_id",
    "base_utilisation",
    "utilisation_scenario_label",
    "is_central_utilisation",
    "flexibility_scenario_id",
    "flexible_workload_share",
    "scenario_label",
    "is_central_flexibility",
    "is_central",
]
BASE_ALLOCATION_COLUMNS = [
    "allocation_method",
    "shift_mode",
    "model_layer",
    "operator_group",
    "weather_year",
    "utc_date",
    "sender_query_location_id",
    "sender_country",
    "sender_country_code",
    "sender_timezone",
    "sender_renewable_query_region",
    "sender_time_utc",
    "sender_utc_hour",
    "sender_renewable_availability_index",
    "receiver_query_location_id",
    "receiver_country",
    "receiver_country_code",
    "receiver_timezone",
    "receiver_renewable_query_region",
    "receiver_time_utc",
    "receiver_utc_hour",
    "receiver_renewable_availability_index",
    "shifted_mwh",
    "renewable_availability_gain",
    "time_difference_hours",
    "weighted_renewable_gain",
    "weighted_abs_time_hours",
]
ALLOCATION_OUTPUT_COLUMNS = SCENARIO_METADATA_COLUMNS + BASE_ALLOCATION_COLUMNS

## Reuse completed allocations

Set `ALLOW_NEW_ALLOCATIONS = False` to use completed partitions and define the four allocation roots for both hour settings and shift modes. This setting stops execution if matching completed allocations are unavailable.


In [4]:
ALLOW_NEW_ALLOCATIONS = False
allocation_roots = (
    (SIX_HOUR_CANDIDATE_HOURS, "daily_spatiotemporal", DAILY_PARQUET_OUTPUT_DIR),
    (SIX_HOUR_CANDIDATE_HOURS, "same_hour_geographic", SAME_HOUR_PARQUET_OUTPUT_DIR),
    (NINE_HOUR_CANDIDATE_HOURS, "daily_spatiotemporal", DAILY_9H_PARQUET_OUTPUT_DIR),
    (NINE_HOUR_CANDIDATE_HOURS, "same_hour_geographic", SAME_HOUR_9H_PARQUET_OUTPUT_DIR),
)

## Read scenarios and hourly inputs

Define readers for the three utilisation levels, three flexibility levels and hourly model input. Combine the scenario definitions into nine cases with an 85% utilisation and 25% flexibility reference, and load the fields needed by both allocation methods.


In [5]:
# Read the utilisation and flexibility definitions and form their nine-case matrix.
def read_scenario_matrix(utilisation_path: Path, flexibility_path: Path):
    """Read and validate the three utilisation and flexibility definitions."""
    utilisation = pd.read_csv(utilisation_path)
    flexibility = pd.read_csv(flexibility_path)
    required_utilisation = {
        "utilisation_scenario_id", "base_utilisation", "scenario_label", "is_central"
    }
    required_flexibility = {
        "flexibility_scenario_id", "flexible_workload_share", "scenario_label", "is_central"
    }
    missing_utilisation = sorted(required_utilisation - set(utilisation.columns))
    missing_flexibility = sorted(required_flexibility - set(flexibility.columns))
    if missing_utilisation:
        raise ValueError(f"Utilisation scenario table is missing: {missing_utilisation}")
    if missing_flexibility:
        raise ValueError(f"Flexibility scenario table is missing: {missing_flexibility}")

    utilisation["base_utilisation"] = pd.to_numeric(
        utilisation["base_utilisation"], errors="raise"
    )
    flexibility["flexible_workload_share"] = pd.to_numeric(
        flexibility["flexible_workload_share"], errors="raise"
    )
    for frame in (utilisation, flexibility):
        frame["is_central"] = frame["is_central"].map(
            lambda value: str(value).strip().lower() == "true"
        )

    utilisation = utilisation.sort_values("base_utilisation").reset_index(drop=True)
    flexibility = flexibility.sort_values("flexible_workload_share").reset_index(drop=True)
    if utilisation["utilisation_scenario_id"].tolist() != ["util_80", "util_85", "util_90"]:
        raise ValueError("Expected utilisation scenarios util_80, util_85, and util_90")
    if flexibility["flexibility_scenario_id"].tolist() != ["flex_20", "flex_25", "flex_30"]:
        raise ValueError("Expected flexibility scenarios flex_20, flex_25, and flex_30")
    if utilisation["is_central"].sum() != 1 or not utilisation.loc[
        utilisation["utilisation_scenario_id"].eq("util_85"), "is_central"
    ].all():
        raise ValueError("The central utilisation case must be util_85")
    if flexibility["is_central"].sum() != 1 or not flexibility.loc[
        flexibility["flexibility_scenario_id"].eq("flex_25"), "is_central"
    ].all():
        raise ValueError("The central flexibility case must be flex_25")

    utilisation = utilisation.rename(
        columns={
            "scenario_label": "utilisation_scenario_label",
            "is_central": "is_central_utilisation",
        }
    )
    flexibility = flexibility.rename(columns={"is_central": "is_central_flexibility"})
    matrix = utilisation.merge(flexibility, how="cross")
    matrix["is_central"] = (
        matrix["is_central_utilisation"] & matrix["is_central_flexibility"]
    )
    if len(matrix) != 9 or matrix["is_central"].sum() != 1:
        raise ValueError("Scenario matrix must contain nine cases and one central case")
    return utilisation, flexibility, matrix


# Load and validate the hourly fields required by both allocation methods.
def read_hourly_model_inputs(path: Path) -> pd.DataFrame:
    """Read only the scenario-neutral fields used by both allocation methods."""
    required = [
        "model_layer", "query_location_id", "operator_group", "country", "country_code",
        "timezone", "renewable_query_region", "weather_year", "time_utc", "utc_date",
        "utc_hour", "renewable_availability_index", "receive_rank_daily", "send_rank_daily",
        "receive_candidate", "send_candidate", "capacity_mw", "normalized_load_shape",
        "receive_utilisation_ceiling",
    ]
    hourly = pd.read_parquet(path, columns=required)
    hourly["time_utc"] = pd.to_datetime(hourly["time_utc"], utc=True, errors="coerce")
    if hourly["time_utc"].isna().any():
        raise ValueError("Some hourly rows have invalid UTC timestamps")
    hourly["utc_date"] = hourly["time_utc"].dt.date
    numeric = [
        "weather_year", "utc_hour", "receive_rank_daily", "send_rank_daily",
        "renewable_availability_index", "capacity_mw", "normalized_load_shape",
        "receive_utilisation_ceiling",
    ]
    for column in numeric:
        hourly[column] = pd.to_numeric(hourly[column], errors="coerce")
    model_values = ["capacity_mw", "normalized_load_shape", "receive_utilisation_ceiling"]
    if hourly[model_values].isna().any().any():
        raise ValueError("Hourly model inputs contain missing load-model values")
    return hourly

## Select candidate hours

Define candidate selection from each location's daily renewable ranks and check the six-hour selection against the supplied flags. Use the same tie order for both settings, so the nine-hour sets contain all six-hour candidates. Check complete UTC days and count selected hours and any overlap between send and receive candidates.


In [6]:
# Select six or nine candidate hours from the supplied daily ranks.
def apply_candidate_hours(hourly_base: pd.DataFrame, candidate_hours: int) -> pd.DataFrame:
    """Select candidate hours from daily ranks using the per-location UTC tie order."""
    if candidate_hours not in (6, 9):
        raise ValueError("Candidate hours must be 6 or 9")
    for side in ("send", "receive"):
        rank = hourly_base[f"{side}_rank_daily"]
        if rank.isna().any() or not (rank.between(1, 24) & rank.eq(np.floor(rank))).all():
            raise ValueError(f"Invalid {side} daily ranks; expected integers 1–24")
        if not hourly_base[f"{side}_candidate"].eq(rank.le(6)).all():
            raise ValueError(f"Hourly {side}_candidate flags must agree with {side}_rank_daily <= 6")
    if candidate_hours == 6:
        return hourly_base
    hourly = hourly_base.copy()
    for side in ("send", "receive"):
        hourly[f"{side}_candidate"] = hourly[f"{side}_rank_daily"].le(candidate_hours)
    # Independent send/receive tie-breaks can select the same hour on both sides.
    # Retain this overlap so the candidate sets remain nested.
    return hourly


# Check complete UTC-day ranks and report candidate counts and tied overlap.
def validate_candidate_hour_counts(hourly_base):
    """Check full UTC days and report the 6h/9h counts, including tied overlap."""
    group_cols = ["model_layer", "operator_group", "query_location_id", "weather_year", "utc_date"]
    grouped = hourly_base.groupby(group_cols, observed=True, dropna=False)
    checks = grouped.agg(rows=("time_utc", "size"), timestamps=("time_utc", "nunique"),
                         send_ranks=("send_rank_daily", "nunique"), receive_ranks=("receive_rank_daily", "nunique"))
    if not checks.eq(24).all().all():
        raise ValueError("Hourly input must contain 24 distinct timestamps and ranks per node/UTC day")
    apply_candidate_hours(hourly_base, 6)
    for hours in (6, 9):
        sends = hourly_base["send_rank_daily"].le(hours)
        receives = hourly_base["receive_rank_daily"].le(hours)
        if int(sends.sum()) != len(checks) * hours or int(receives.sum()) != len(checks) * hours:
            raise ValueError(f"Unexpected {hours}h candidate counts")
        print(f"{hours}h: {int(sends.sum()):,} send and {int(receives.sum()):,} receive node-hours; "
              f"{int((sends & receives).sum()):,} overlap rows from the daily rank tie order.")

## Calculate load and group candidate hours

Define actual load as the smaller of capacity and capacity × utilisation × normalised shape, with receiver headroom equal to the positive difference between 95% of capacity and actual load. Group and order candidate senders and receivers within each operator, model layer and weather year. Daily groups share a UTC date, while same-hour groups share an exact UTC timestamp.


In [7]:
# Calculate capped hourly load and receiver headroom for one utilisation level.
def apply_utilisation(hourly_base: pd.DataFrame, base_utilisation: float) -> pd.DataFrame:
    """Recalculate load and receiver headroom for one utilisation case."""
    hourly = hourly_base.copy()
    hourly["actual_total_load_mw_uncapped"] = (
        hourly["capacity_mw"] * float(base_utilisation) * hourly["normalized_load_shape"]
    )
    hourly["actual_total_load_mw"] = np.minimum(
        hourly["actual_total_load_mw_uncapped"], hourly["capacity_mw"]
    )
    hourly["load_capped"] = hourly["actual_total_load_mw_uncapped"].gt(
        hourly["capacity_mw"]
    )
    hourly["receive_headroom_mw"] = (
        hourly["capacity_mw"] * hourly["receive_utilisation_ceiling"]
        - hourly["actual_total_load_mw"]
    ).clip(lower=0)
    hourly["eligible_receive_mwh"] = np.where(
        hourly["receive_candidate"].astype(bool), hourly["receive_headroom_mw"], 0.0
    )
    if hourly["actual_total_load_mw"].gt(hourly["capacity_mw"] + FEASIBILITY_TOLERANCE_MWH).any():
        raise ValueError("Actual load exceeds installed capacity")
    if hourly["receive_headroom_mw"].lt(-FEASIBILITY_TOLERANCE_MWH).any():
        raise ValueError("Receiver headroom is negative")
    return hourly


# Build deterministically ordered sender and receiver groups for the selected shift mode.
def _group_hourly_for_mode(hourly: pd.DataFrame, shift_mode: str):
    send_base = hourly[
        hourly["send_candidate"].astype(bool) & hourly["actual_total_load_mw"].gt(0)
    ].copy()
    receive_base = hourly[
        hourly["receive_candidate"].astype(bool) & hourly["eligible_receive_mwh"].gt(0)
    ].copy()
    if shift_mode == "daily_spatiotemporal":
        group_cols = ["model_layer", "operator_group", "weather_year", "utc_date"]
        send_sort = ["send_rank_daily", "renewable_availability_index", "time_utc", "query_location_id"]
        send_ascending = [True, True, True, True]
        receive_sort = ["receive_rank_daily", "renewable_availability_index", "time_utc", "query_location_id"]
        receive_ascending = [True, False, True, True]
    elif shift_mode == "same_hour_geographic":
        group_cols = ["model_layer", "operator_group", "weather_year", "time_utc"]
        send_sort = ["renewable_availability_index", "query_location_id"]
        send_ascending = [True, True]
        receive_sort = ["renewable_availability_index", "query_location_id"]
        receive_ascending = [False, True]
    else:
        raise ValueError(f"Unsupported shift mode: {shift_mode}")

    send_by_group = {
        key: group.sort_values(send_sort, ascending=send_ascending).reset_index(drop=True)
        for key, group in send_base.groupby(group_cols, dropna=False, sort=False)
    }
    receive_by_group = {
        key: group.sort_values(receive_sort, ascending=receive_ascending).reset_index(drop=True)
        for key, group in receive_base.groupby(group_cols, dropna=False, sort=False)
    }
    common_keys = sorted(set(send_by_group).intersection(receive_by_group))
    return send_by_group, receive_by_group, common_keys

## Define allocation records

Define the scenario metadata and temporal identifiers included in each allocation record. Record sender and receiver locations, shifted energy, renewable gain and absolute time difference for each transfer.


In [8]:
# Select the scenario labels and flags carried by every result.
def _scenario_metadata(scenario: dict) -> dict:
    return {column: scenario[column] for column in SCENARIO_METADATA_COLUMNS}


# Convert the temporal group key into the shared result dimensions.
def _group_dimensions(key, sends: pd.DataFrame, shift_mode: str) -> dict:
    if shift_mode == "daily_spatiotemporal":
        model_layer, operator_group, weather_year, utc_date = key
    elif shift_mode == "same_hour_geographic":
        model_layer, operator_group, weather_year, time_utc = key
        utc_date = sends["utc_date"].iloc[0]
    else:
        raise ValueError(f"Unsupported shift mode: {shift_mode}")
    return {
        "model_layer": model_layer,
        "operator_group": operator_group,
        "weather_year": int(weather_year),
        "utc_date": utc_date,
    }


# Create one flow record with provenance, shifted energy, gain and time difference.
def _allocation_record(
    *, metadata: dict, allocation_method: str, shift_mode: str, dimensions: dict,
    send, receives: pd.DataFrame, receiver_index: int, amount: float,
) -> dict:
    receiver = receives.iloc[receiver_index]
    gain = float(receiver["renewable_availability_index"] - send.renewable_availability_index)
    time_difference = abs(
        (receiver["time_utc"] - send.time_utc).total_seconds()
    ) / 3600
    return {
        **metadata,
        "allocation_method": allocation_method,
        "shift_mode": shift_mode,
        **dimensions,
        "sender_query_location_id": send.query_location_id,
        "sender_country": send.country,
        "sender_country_code": send.country_code,
        "sender_timezone": send.timezone,
        "sender_renewable_query_region": send.renewable_query_region,
        "sender_time_utc": send.time_utc,
        "sender_utc_hour": int(send.utc_hour),
        "sender_renewable_availability_index": float(send.renewable_availability_index),
        "receiver_query_location_id": receiver["query_location_id"],
        "receiver_country": receiver["country"],
        "receiver_country_code": receiver["country_code"],
        "receiver_timezone": receiver["timezone"],
        "receiver_renewable_query_region": receiver["renewable_query_region"],
        "receiver_time_utc": receiver["time_utc"],
        "receiver_utc_hour": int(receiver["utc_hour"]),
        "receiver_renewable_availability_index": float(receiver["renewable_availability_index"]),
        "shifted_mwh": float(amount),
        "renewable_availability_gain": gain,
        "time_difference_hours": time_difference,
        "weighted_renewable_gain": float(amount) * gain,
        "weighted_abs_time_hours": float(amount) * time_difference,
    }

## Allocate with the greedy method

Define the greedy allocator to visit senders and receivers in a fixed order. Limit each sender to actual load × flexible share and reduce the remaining sender supply and receiver headroom after each transfer. Allow transfers only to a different query location with a higher renewable availability index.


In [9]:
def _allocate_group_greedy(key, sends, receives, scenario, shift_mode: str):
    """Allocate in deterministic order, decrementing sender potential and headroom."""
    metadata = _scenario_metadata(scenario)
    dimensions = _group_dimensions(key, sends, shift_mode)
    flexible_share = float(scenario["flexible_workload_share"])
    receive_remaining = receives["eligible_receive_mwh"].to_numpy(dtype=float).copy()
    receiver_qids = receives["query_location_id"].astype(str).to_numpy()
    receiver_index_values = receives["renewable_availability_index"].to_numpy(dtype=float)
    allocations = []

    for send in sends.itertuples(index=False):
        send_remaining = float(send.actual_total_load_mw) * flexible_share
        if send_remaining <= ALLOCATION_TOLERANCE_MWH:
            continue
        send_qid = str(send.query_location_id)
        send_index = float(send.renewable_availability_index)
        eligible = np.flatnonzero(
            (receive_remaining > ALLOCATION_TOLERANCE_MWH)
            & (receiver_index_values - send_index > ALLOCATION_TOLERANCE_MWH)
            & (receiver_qids != send_qid)
        )
        for receiver_index in eligible:
            if send_remaining <= ALLOCATION_TOLERANCE_MWH:
                break
            amount = min(send_remaining, float(receive_remaining[receiver_index]))
            if amount <= ALLOCATION_TOLERANCE_MWH:
                continue
            allocations.append(
                _allocation_record(
                    metadata=metadata,
                    allocation_method="greedy",
                    shift_mode=shift_mode,
                    dimensions=dimensions,
                    send=send,
                    receives=receives,
                    receiver_index=int(receiver_index),
                    amount=amount,
                )
            )
            send_remaining -= amount
            receive_remaining[receiver_index] -= amount
    return allocations

## Construct the feasible network

Define the LP network using the same positive-gain transfers between different locations available to the greedy method. Return sender supply, receiver headroom and each feasible edge's renewable gain and time difference.


In [10]:
def _feasible_network(sends, receives, flexible_share: float):
    """Construct the shared sparse bipartite network for one temporal group."""
    supply = sends["actual_total_load_mw"].to_numpy(dtype=float) * float(flexible_share)
    headroom = receives["eligible_receive_mwh"].to_numpy(dtype=float)
    send_qids = sends["query_location_id"].astype(str).to_numpy()
    receive_qids = receives["query_location_id"].astype(str).to_numpy()
    send_index = sends["renewable_availability_index"].to_numpy(dtype=float)
    receive_index = receives["renewable_availability_index"].to_numpy(dtype=float)

    sender_index = np.repeat(np.arange(len(sends), dtype=np.int32), len(receives))
    receiver_index = np.tile(np.arange(len(receives), dtype=np.int32), len(sends))
    gains = receive_index[receiver_index] - send_index[sender_index]
    feasible = (
        (supply[sender_index] > ALLOCATION_TOLERANCE_MWH)
        & (headroom[receiver_index] > ALLOCATION_TOLERANCE_MWH)
        & (gains > ALLOCATION_TOLERANCE_MWH)
        & (send_qids[sender_index] != receive_qids[receiver_index])
    )
    sender_index = sender_index[feasible]
    receiver_index = receiver_index[feasible]
    gains = gains[feasible]
    if len(sender_index):
        sender_times = sends["time_utc"].astype("int64").to_numpy()[sender_index]
        receiver_times = receives["time_utc"].astype("int64").to_numpy()[receiver_index]
        time_differences = np.abs(receiver_times - sender_times) / 3_600_000_000_000
    else:
        time_differences = np.array([], dtype=float)
    return supply, headroom, sender_index, receiver_index, gains, time_differences

## Solve LP stages within tolerances

Define the absolute and relative tolerance bands used to preserve earlier LP objectives. Add these bands as sparse inequalities and define a HiGHS solver call that rejects unsuccessful results.


In [11]:
# Calculate the allowed absolute or relative change in an earlier LP objective.
def _objective_tolerance(value: float) -> float:
    """Return the declared absolute/relative lexicographic preservation band."""
    return max(
        LEXICOGRAPHIC_ABSOLUTE_TOLERANCE,
        abs(float(value)) * LEXICOGRAPHIC_RELATIVE_TOLERANCE,
    )


# Add sparse inequalities that preserve an earlier objective within its tolerance.
def _add_objective_band(a_ub, b_ub, objective_row, optimum: float, tolerance: float):
    """Append lower and upper sparse inequalities around an earlier optimum."""
    objective_row = csr_matrix(objective_row)
    bounded_a = vstack([a_ub, -objective_row, objective_row], format="csr")
    bounded_b = np.concatenate(
        [b_ub, np.array([-(optimum - tolerance), optimum + tolerance])]
    )
    return bounded_a, bounded_b


# Solve one LP stage and reject an unsuccessful solver result.
def _solve_stage(c, *, a_ub, b_ub, label: str):
    started = time.perf_counter()
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"Unrecognized options detected:.*threads.*",
            category=OptimizeWarning,
        )
        result = linprog(
            c,
            A_ub=a_ub,
            b_ub=b_ub,
            bounds=(0, None),
            method="highs-ipm",
            options=SOLVER_OPTIONS,
        )
    elapsed = time.perf_counter() - started
    if result.status != 0 or not result.success:
        raise RuntimeError(
            f"{label} optimisation failed with status {result.status}: {result.message}"
        )
    return result, elapsed

## Batch independent temporal groups

Combine independent temporal groups into ordered batches within one UTC date, starting a new batch when adding a group would exceed the edge limit. Define status records for groups without feasible edges, which do not require a solver call.


In [12]:
# Obtain the UTC date used to keep each batch within one day.
def _temporal_group_date(key, shift_mode: str):
    if shift_mode == "daily_spatiotemporal":
        return key[-1]
    return pd.Timestamp(key[-1]).date()


# Assemble independent temporal groups into ordered sparse batches.
def _iter_optimisation_batches(
    common_keys,
    send_by_group,
    receive_by_group,
    scenario,
    shift_mode: str,
):
    """Yield UTC-date-bounded batches below the declared sparse edge cap.

    The temporal groups are independent.  Solving their block-diagonal union is
    exactly separable: a batch-wide maximum equals the sum of group maxima, and
    no group can compensate for another group falling below its own maximum.
    The same argument applies to each later objective once earlier optima are
    preserved.  Post-solve group checks protect that equivalence numerically.
    """
    current_batch = []
    current_date = None
    current_edges = 0
    flexible_share = float(scenario["flexible_workload_share"])
    if shift_mode == "daily_spatiotemporal":
        ordered_keys = sorted(
            common_keys,
            key=lambda key: (key[-1], str(key[0]), str(key[1]), int(key[2])),
        )
    else:
        ordered_keys = sorted(
            common_keys,
            key=lambda key: (
                pd.Timestamp(key[-1]).date(),
                pd.Timestamp(key[-1]),
                str(key[0]),
                str(key[1]),
                int(key[2]),
            ),
        )
    for key in ordered_keys:
        sends = send_by_group[key]
        receives = receive_by_group[key]
        network = _feasible_network(sends, receives, flexible_share)
        edge_count = len(network[2])
        group_date = _temporal_group_date(key, shift_mode)
        starts_new_batch = current_batch and (
            group_date != current_date
            or current_edges + edge_count > OPTIMISATION_BATCH_EDGE_LIMIT
        )
        if starts_new_batch:
            yield current_batch
            current_batch = []
            current_edges = 0
        current_batch.append(
            {
                "key": key,
                "sends": sends,
                "receives": receives,
                "network": network,
            }
        )
        current_date = group_date
        current_edges += edge_count
    if current_batch:
        yield current_batch


# Record a temporal group with no feasible edges without invoking the solver.
def _empty_optimisation_status(group, shift_mode: str, batch_id: str):
    dimensions = _group_dimensions(group["key"], group["sends"], shift_mode)
    return {
        **dimensions,
        "allocation_method": "optimised_lp",
        "shift_mode": shift_mode,
        "solver_batch_id": batch_id,
        "sender_count": len(group["sends"]),
        "receiver_count": len(group["receives"]),
        "feasible_edge_count": 0,
        "has_feasible_edges": False,
        "stage1_status": "not_run_no_feasible_edges",
        "stage2_status": "not_run_no_feasible_edges",
        "stage3_status": "not_run_no_feasible_edges",
        "stage1_optimal_shifted_mwh": 0.0,
        "stage2_optimal_weighted_gain": 0.0,
        "stage3_optimal_weighted_abs_time_hours": 0.0,
        "final_shifted_mwh": 0.0,
        "final_weighted_renewable_gain": 0.0,
        "final_weighted_abs_time_hours": 0.0,
        "stage1_preservation_tolerance_mwh": 0.0,
        "stage2_preservation_tolerance_weighted_gain": 0.0,
        "final_minus_stage1_shifted_mwh": 0.0,
        "final_minus_stage2_weighted_gain": 0.0,
        "total_solver_seconds": 0.0,
    }

## Allocate with linear programming

Define the LP allocator to maximise shifted energy first and renewable gain second, preserving earlier objectives within their numerical tolerances. For daily shifting, add a third stage that minimises absolute time difference. Check sender limits, receiver headroom and objective preservation before recording the resulting flows.


In [13]:
def _allocate_batch_optimised(
    batch,
    scenario,
    shift_mode: str,
    batch_sequence: int,
):
    """Solve one sparse block-diagonal batch with objective-preservation bands."""
    metadata = _scenario_metadata(scenario)
    batch_date = _temporal_group_date(batch[0]["key"], shift_mode)
    batch_id = f"{shift_mode}:{batch_date}:batch_{batch_sequence:04d}"
    active_groups = []
    empty_statuses = []
    for group in batch:
        if len(group["network"][2]) == 0:
            empty_statuses.append(
                _empty_optimisation_status(group, shift_mode, batch_id)
            )
        else:
            active_groups.append(group)
    if not active_groups:
        return [], empty_statuses

    constraint_matrices = []
    constraint_bounds = []
    gains_by_group = []
    time_by_group = []
    edge_slices = []
    edge_offset = 0
    for group in active_groups:
        supply, headroom, sender_index, receiver_index, gains, time_differences = group["network"]
        edge_count = len(sender_index)
        edge_ids = np.arange(edge_count, dtype=np.int32)
        constraint_rows = np.concatenate(
            [sender_index, len(supply) + receiver_index]
        )
        constraint_cols = np.concatenate([edge_ids, edge_ids])
        group_constraints = coo_matrix(
            (
                np.ones(2 * edge_count),
                (constraint_rows, constraint_cols),
            ),
            shape=(len(supply) + len(headroom), edge_count),
        ).tocsr()
        constraint_matrices.append(group_constraints)
        constraint_bounds.append(np.concatenate([supply, headroom]))
        gains_by_group.append(gains)
        time_by_group.append(time_differences)
        edge_slices.append(slice(edge_offset, edge_offset + edge_count))
        edge_offset += edge_count

    base_a_ub = block_diag(constraint_matrices, format="csr")
    base_b_ub = np.concatenate(constraint_bounds)
    gains_all = np.concatenate(gains_by_group)
    time_all = np.concatenate(time_by_group)
    sum_row = csr_matrix(np.ones((1, edge_offset), dtype=float))

    stage1, seconds1 = _solve_stage(
        -np.ones(edge_offset),
        a_ub=base_a_ub,
        b_ub=base_b_ub,
        label=f"Stage 1 ({batch_id})",
    )
    optimal_volume = float(stage1.x.sum())
    volume_tolerance = _objective_tolerance(optimal_volume)
    stage2_a_ub, stage2_b_ub = _add_objective_band(
        base_a_ub,
        base_b_ub,
        sum_row,
        optimal_volume,
        volume_tolerance,
    )
    stage2, seconds2 = _solve_stage(
        -gains_all,
        a_ub=stage2_a_ub,
        b_ub=stage2_b_ub,
        label=f"Stage 2 ({batch_id})",
    )
    optimal_gain = float(np.dot(stage2.x, gains_all))
    gain_tolerance = _objective_tolerance(optimal_gain)

    if shift_mode == "daily_spatiotemporal":
        gain_row = csr_matrix(gains_all.reshape(1, -1))
        stage3_a_ub, stage3_b_ub = _add_objective_band(
            stage2_a_ub,
            stage2_b_ub,
            gain_row,
            optimal_gain,
            gain_tolerance,
        )
        stage3, seconds3 = _solve_stage(
            time_all,
            a_ub=stage3_a_ub,
            b_ub=stage3_b_ub,
            label=f"Stage 3 ({batch_id})",
        )
        final_x = stage3.x
        stage3_status = "optimal"
    else:
        final_x = stage2.x
        seconds3 = 0.0
        stage3_status = "not_applicable"

    final_volume = float(final_x.sum())
    final_gain = float(np.dot(final_x, gains_all))
    if abs(final_volume - optimal_volume) > volume_tolerance + FEASIBILITY_TOLERANCE_MWH:
        raise RuntimeError(f"{batch_id} did not preserve stage-1 shifted MWh")
    if abs(final_gain - optimal_gain) > gain_tolerance + FEASIBILITY_TOLERANCE_MWH:
        raise RuntimeError(f"{batch_id} did not preserve stage-2 renewable gain")

    allocations = []
    statuses = list(empty_statuses)
    seconds_share = (seconds1 + seconds2 + seconds3) / len(active_groups)
    for group, edge_slice in zip(active_groups, edge_slices):
        sends = group["sends"]
        receives = group["receives"]
        supply, headroom, sender_index, receiver_index, gains, time_differences = group["network"]
        group_stage1_x = stage1.x[edge_slice]
        group_stage2_x = stage2.x[edge_slice]
        group_final_x = final_x[edge_slice]
        group_stage1_volume = float(group_stage1_x.sum())
        group_stage2_gain = float(np.dot(group_stage2_x, gains))
        group_final_volume = float(group_final_x.sum())
        group_final_gain = float(np.dot(group_final_x, gains))
        group_final_time = float(np.dot(group_final_x, time_differences))
        sender_use = np.bincount(
            sender_index, weights=group_final_x, minlength=len(sends)
        )
        receiver_use = np.bincount(
            receiver_index, weights=group_final_x, minlength=len(receives)
        )
        if (sender_use - supply > FEASIBILITY_TOLERANCE_MWH).any():
            raise RuntimeError(f"{batch_id} flow exceeds sender potential")
        if (receiver_use - headroom > FEASIBILITY_TOLERANCE_MWH).any():
            raise RuntimeError(f"{batch_id} flow exceeds receiver headroom")
        if group_final_volume + volume_tolerance + FEASIBILITY_TOLERANCE_MWH < group_stage1_volume:
            raise RuntimeError(f"{batch_id} did not preserve a group's shifted-MWh optimum")
        if group_final_gain + gain_tolerance + FEASIBILITY_TOLERANCE_MWH < group_stage2_gain:
            raise RuntimeError(f"{batch_id} did not preserve a group's renewable-gain optimum")

        dimensions = _group_dimensions(group["key"], sends, shift_mode)
        send_tuples = list(sends.itertuples(index=False))
        for edge in np.flatnonzero(group_final_x > ALLOCATION_TOLERANCE_MWH):
            allocations.append(
                _allocation_record(
                    metadata=metadata,
                    allocation_method="optimised_lp",
                    shift_mode=shift_mode,
                    dimensions=dimensions,
                    send=send_tuples[int(sender_index[edge])],
                    receives=receives,
                    receiver_index=int(receiver_index[edge]),
                    amount=float(group_final_x[edge]),
                )
            )
        statuses.append(
            {
                **dimensions,
                "allocation_method": "optimised_lp",
                "shift_mode": shift_mode,
                "solver_batch_id": batch_id,
                "sender_count": len(sends),
                "receiver_count": len(receives),
                "feasible_edge_count": len(sender_index),
                "has_feasible_edges": True,
                "stage1_status": "optimal",
                "stage2_status": "optimal",
                "stage3_status": stage3_status,
                "stage1_optimal_shifted_mwh": group_stage1_volume,
                "stage2_optimal_weighted_gain": group_stage2_gain,
                "stage3_optimal_weighted_abs_time_hours": group_final_time,
                "final_shifted_mwh": group_final_volume,
                "final_weighted_renewable_gain": group_final_gain,
                "final_weighted_abs_time_hours": group_final_time,
                "stage1_preservation_tolerance_mwh": volume_tolerance,
                "stage2_preservation_tolerance_weighted_gain": gain_tolerance,
                "final_minus_stage1_shifted_mwh": group_final_volume - group_stage1_volume,
                "final_minus_stage2_weighted_gain": group_final_gain - group_stage2_gain,
                "total_solver_seconds": seconds_share,
            }
        )
    return allocations, statuses

## Summarise energy and workload

Define the load totals, capacity-hour totals and energy-weighted renewable and time metrics used by both allocation methods. Summarise transfers by operator and UTC day, then by operator and weather year, retaining operator-year groups with no transfers.


In [14]:
# Aggregate operator/year load and capacity-hour denominators for one scenario.
def _load_totals(hourly: pd.DataFrame, scenario: dict) -> pd.DataFrame:
    grouped = (
        hourly.groupby(["model_layer", "operator_group", "weather_year"], dropna=False)
        .agg(
            total_actual_load_mwh=("actual_total_load_mw", "sum"),
            total_capacity_hours_mwh=("capacity_mw", "sum"),
            capped_hour_count=("load_capped", "sum"),
            hourly_observation_count=("time_utc", "size"),
            days_covered=("utc_date", "nunique"),
        )
        .reset_index()
    )
    grouped["realised_base_utilisation"] = (
        grouped["total_actual_load_mwh"] / grouped["total_capacity_hours_mwh"]
    )
    for column, value in _scenario_metadata(scenario).items():
        grouped[column] = value
    return grouped


# Calculate shifted-energy-weighted renewable gain and absolute time difference.
def _weighted_metrics(frame: pd.DataFrame) -> pd.DataFrame:
    frame["mwh_weighted_renewable_availability_gain"] = np.divide(
        frame["total_weighted_renewable_gain"],
        frame["shifted_mwh"],
        out=np.zeros(len(frame), dtype=float),
        where=frame["shifted_mwh"].to_numpy(dtype=float) > ALLOCATION_TOLERANCE_MWH,
    )
    frame["mwh_weighted_absolute_time_difference_hours"] = np.divide(
        frame["total_weighted_abs_time_hours"],
        frame["shifted_mwh"],
        out=np.zeros(len(frame), dtype=float),
        where=frame["shifted_mwh"].to_numpy(dtype=float) > ALLOCATION_TOLERANCE_MWH,
    )
    return frame


# Build daily and operator/year summaries while retaining zero-transfer operator/year groups.
def summarize_allocations(
    allocations: pd.DataFrame,
    load_totals: pd.DataFrame,
    *,
    allocation_method: str,
    shift_mode: str,
):
    daily_group_cols = SCENARIO_METADATA_COLUMNS + [
        "allocation_method", "shift_mode", "model_layer", "operator_group", "weather_year", "utc_date"
    ]
    if allocations.empty:
        daily = pd.DataFrame(columns=daily_group_cols + [
            "allocation_count", "shifted_mwh", "sent_mwh", "received_mwh",
            "sender_location_count", "receiver_location_count",
            "total_weighted_renewable_gain", "total_weighted_abs_time_hours",
            "mwh_weighted_renewable_availability_gain",
            "mwh_weighted_absolute_time_difference_hours",
        ])
        allocation_operator_year = pd.DataFrame()
    else:
        daily = (
            allocations.groupby(daily_group_cols, dropna=False)
            .agg(
                allocation_count=("shifted_mwh", "size"),
                shifted_mwh=("shifted_mwh", "sum"),
                sent_mwh=("shifted_mwh", "sum"),
                received_mwh=("shifted_mwh", "sum"),
                sender_location_count=("sender_query_location_id", "nunique"),
                receiver_location_count=("receiver_query_location_id", "nunique"),
                total_weighted_renewable_gain=("weighted_renewable_gain", "sum"),
                total_weighted_abs_time_hours=("weighted_abs_time_hours", "sum"),
            )
            .reset_index()
        )
        daily = _weighted_metrics(daily)
        operator_group_cols = SCENARIO_METADATA_COLUMNS + [
            "allocation_method", "shift_mode", "model_layer", "operator_group", "weather_year"
        ]
        allocation_operator_year = (
            daily.groupby(operator_group_cols, dropna=False)
            .agg(
                active_days=("utc_date", "nunique"),
                allocation_count=("allocation_count", "sum"),
                shifted_mwh=("shifted_mwh", "sum"),
                average_daily_shifted_mwh=("shifted_mwh", "mean"),
                median_daily_shifted_mwh=("shifted_mwh", "median"),
                max_daily_shifted_mwh=("shifted_mwh", "max"),
                total_weighted_renewable_gain=("total_weighted_renewable_gain", "sum"),
                total_weighted_abs_time_hours=("total_weighted_abs_time_hours", "sum"),
            )
            .reset_index()
        )

    operator_year = load_totals.copy()
    operator_year["allocation_method"] = allocation_method
    operator_year["shift_mode"] = shift_mode
    merge_cols = SCENARIO_METADATA_COLUMNS + [
        "allocation_method", "shift_mode", "model_layer", "operator_group", "weather_year"
    ]
    if allocation_operator_year.empty:
        for column in [
            "active_days", "allocation_count", "shifted_mwh", "average_daily_shifted_mwh",
            "median_daily_shifted_mwh", "max_daily_shifted_mwh",
            "total_weighted_renewable_gain", "total_weighted_abs_time_hours",
        ]:
            operator_year[column] = 0.0
    else:
        operator_year = operator_year.merge(
            allocation_operator_year, on=merge_cols, how="left", validate="one_to_one"
        )
        fill_zero = [
            "active_days", "allocation_count", "shifted_mwh", "average_daily_shifted_mwh",
            "median_daily_shifted_mwh", "max_daily_shifted_mwh",
            "total_weighted_renewable_gain", "total_weighted_abs_time_hours",
        ]
        operator_year[fill_zero] = operator_year[fill_zero].fillna(0)
    operator_year = _weighted_metrics(operator_year)
    total_by_year = (
        operator_year.groupby(
            SCENARIO_METADATA_COLUMNS + ["allocation_method", "shift_mode", "model_layer", "weather_year"],
            dropna=False,
        )["shifted_mwh"]
        .sum()
        .reset_index(name="model_layer_weather_year_shifted_mwh")
    )
    operator_year = operator_year.merge(
        total_by_year,
        on=SCENARIO_METADATA_COLUMNS + ["allocation_method", "shift_mode", "model_layer", "weather_year"],
        how="left",
        validate="many_to_one",
    )
    operator_year["operator_share_of_layer_year_shifted_mwh"] = np.divide(
        operator_year["shifted_mwh"],
        operator_year["model_layer_weather_year_shifted_mwh"],
        out=np.zeros(len(operator_year), dtype=float),
        where=operator_year["model_layer_weather_year_shifted_mwh"].to_numpy(dtype=float) > ALLOCATION_TOLERANCE_MWH,
    )
    return daily, operator_year

## Summarise solver results

Aggregate solver records by operator and weather year to count solved groups and groups without feasible edges. Retain objective totals, preservation errors and tolerance values for the later allocation comparisons.


In [15]:
def _aggregate_solver_status(status_records: list[dict], scenario: dict) -> pd.DataFrame:
    status = pd.DataFrame(status_records)
    if status.empty:
        return status
    group_cols = ["allocation_method", "shift_mode", "model_layer", "operator_group", "weather_year"]
    status["stage1_optimal"] = status["stage1_status"].eq("optimal")
    status["stage2_optimal"] = status["stage2_status"].eq("optimal")
    status["stage3_optimal"] = status["stage3_status"].eq("optimal")
    status["stage3_not_applicable"] = status["stage3_status"].eq("not_applicable")
    aggregated = (
        status.groupby(group_cols, dropna=False)
        .agg(
            temporal_group_count=("utc_date", "size"),
            optimisation_batch_count=("solver_batch_id", "nunique"),
            feasible_edge_group_count=("has_feasible_edges", "sum"),
            feasible_edge_count=("feasible_edge_count", "sum"),
            stage1_optimal_group_count=("stage1_optimal", "sum"),
            stage2_optimal_group_count=("stage2_optimal", "sum"),
            stage3_optimal_group_count=("stage3_optimal", "sum"),
            stage3_not_applicable_group_count=("stage3_not_applicable", "sum"),
            stage1_optimal_shifted_mwh=("stage1_optimal_shifted_mwh", "sum"),
            stage2_optimal_weighted_gain=("stage2_optimal_weighted_gain", "sum"),
            stage3_optimal_weighted_abs_time_hours=("stage3_optimal_weighted_abs_time_hours", "sum"),
            final_shifted_mwh=("final_shifted_mwh", "sum"),
            final_weighted_renewable_gain=("final_weighted_renewable_gain", "sum"),
            final_weighted_abs_time_hours=("final_weighted_abs_time_hours", "sum"),
            maximum_stage1_preservation_tolerance_mwh=("stage1_preservation_tolerance_mwh", "max"),
            maximum_stage2_preservation_tolerance_weighted_gain=("stage2_preservation_tolerance_weighted_gain", "max"),
            maximum_absolute_stage1_preservation_error_mwh=("final_minus_stage1_shifted_mwh", lambda values: values.abs().max()),
            maximum_absolute_stage2_preservation_error_weighted_gain=("final_minus_stage2_weighted_gain", lambda values: values.abs().max()),
            total_solver_seconds=("total_solver_seconds", "sum"),
        )
        .reset_index()
    )
    aggregated["no_feasible_edge_group_count"] = (
        aggregated["temporal_group_count"] - aggregated["feasible_edge_group_count"]
    )
    for column, value in _scenario_metadata(scenario).items():
        aggregated[column] = value
    return aggregated[SCENARIO_METADATA_COLUMNS + [
        "allocation_method", "shift_mode", "model_layer", "operator_group", "weather_year",
        "temporal_group_count", "optimisation_batch_count", "feasible_edge_group_count",
        "no_feasible_edge_group_count", "feasible_edge_count", "stage1_optimal_group_count", "stage2_optimal_group_count",
        "stage3_optimal_group_count", "stage3_not_applicable_group_count",
        "stage1_optimal_shifted_mwh", "stage2_optimal_weighted_gain",
        "stage3_optimal_weighted_abs_time_hours", "final_shifted_mwh",
        "final_weighted_renewable_gain", "final_weighted_abs_time_hours",
        "maximum_stage1_preservation_tolerance_mwh",
        "maximum_stage2_preservation_tolerance_weighted_gain",
        "maximum_absolute_stage1_preservation_error_mwh",
        "maximum_absolute_stage2_preservation_error_weighted_gain",
        "total_solver_seconds",
    ]]

## Identify allocation roots and partitions

Define file hashes, root manifests and partition identities to match saved allocations to their inputs and settings. Define recovery of interrupted matching runs and reject incompatible data in an existing output folder.


In [16]:
# Hash file contents to identify the inputs and completed partition files.
def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Describe the input hashes and configuration required to reuse an allocation root.
def _root_manifest(hourly_path, utilisation_path, flexibility_path, shift_mode, candidate_hours=6):
    manifest = {
        "implementation_id": MODEL_IMPLEMENTATION_ID,
        "allocation_methods": list(ALLOCATION_METHODS),
        "shift_mode": shift_mode,
        "scipy_version": scipy.__version__,
        "optimisation_batch_edge_limit": OPTIMISATION_BATCH_EDGE_LIMIT,
        "optimisation_max_workers": OPTIMISATION_MAX_WORKERS,
        "highs_threads_per_worker": HIGHS_THREADS_PER_WORKER,
        "lexicographic_absolute_tolerance": LEXICOGRAPHIC_ABSOLUTE_TOLERANCE,
        "lexicographic_relative_tolerance": LEXICOGRAPHIC_RELATIVE_TOLERANCE,
        "hourly_input_sha256": _sha256_file(hourly_path),
        "utilisation_scenarios_sha256": _sha256_file(utilisation_path),
        "flexibility_scenarios_sha256": _sha256_file(flexibility_path),
    }
    # Six-hour partitions use the schema without candidate-hour metadata.
    if candidate_hours != 6:
        manifest.update(candidate_hours=int(candidate_hours), eligibility_rule="frozen-daily-ranks-v1")
    return manifest


# Reuse or recover a matching root and preserve incompatible existing data.
def _prepare_output_root(path: Path, expected_manifest: dict):
    """Create a fresh allocation root or recover an interrupted matching run."""
    manifest_path = path / "_allocation_root_manifest.json"
    existing = None
    if manifest_path.exists():
        try:
            existing = json.loads(manifest_path.read_text())
        except json.JSONDecodeError:
            existing = None
    if path.exists() and any(path.iterdir()) and existing != expected_manifest:
        raise RuntimeError(f"Allocation cache does not match the requested inputs/settings: {path}. "
                           "Existing files were preserved; use a separate empty output directory "
                           "to calculate this configuration.")
    recovery_needed = any(item.is_dir() for item in path.rglob(".tmp-*")) or any(
        item.is_dir() for item in path.rglob(".backup-*")
    )
    if existing == expected_manifest and not recovery_needed:
        return
    path.mkdir(parents=True, exist_ok=True)
    for temporary in path.rglob(".tmp-*"):
        if temporary.is_dir():
            shutil.rmtree(temporary)
    for backup in path.rglob(".backup-*"):
        if backup.is_dir():
            restored_name = backup.name.split("-", 2)[1]
            restored = backup.parent / restored_name
            if not restored.exists():
                backup.replace(restored)
            else:
                shutil.rmtree(backup)
    if existing != expected_manifest:
        temporary_manifest = manifest_path.with_suffix(".json.tmp")
        temporary_manifest.write_text(json.dumps(expected_manifest, indent=2, sort_keys=True) + "\n")
        temporary_manifest.replace(manifest_path)


# Locate one method, utilisation and flexibility partition below its allocation root.
def _partition_path(root: Path, allocation_method: str, util_id: str, flex_id: str) -> Path:
    return root / f"method_{allocation_method}" / f"util_{util_id}" / f"flex_{flex_id}"


# Define the scenario and method identity required for a completed partition.
def _partition_contract(allocation_method, scenario, shift_mode, candidate_hours=6):
    contract = {
        "implementation_id": MODEL_IMPLEMENTATION_ID,
        "allocation_method": allocation_method,
        "shift_mode": shift_mode,
        "utilisation_scenario_id": scenario["utilisation_scenario_id"],
        "flexibility_scenario_id": scenario["flexibility_scenario_id"],
    }
    if candidate_hours != 6:
        contract.update(candidate_hours=int(candidate_hours), eligibility_rule="frozen-daily-ranks-v1")
    return contract


# Select the detailed-output columns declared by the partition metadata.
def _detail_columns(contract):
    return ALLOCATION_OUTPUT_COLUMNS + (["candidate_hours"] if "candidate_hours" in contract else [])

## Read and write completed partitions

Define a reader that checks partition identity, checksums and schema before accepting completed allocations. Define a writer that prepares complete files in a temporary folder before replacing a partition, with recovery if replacement fails.


In [17]:
# Read a completed partition only when its identity, checksums and schema match.
def _load_completed_partition(partition: Path, contract: dict):
    success_path = partition / "_SUCCESS.json"
    detail_path = partition / "part-00000.parquet"
    if not success_path.exists() or not detail_path.exists():
        return None
    try:
        success = json.loads(success_path.read_text())
    except json.JSONDecodeError:
        return None
    for key, value in contract.items():
        if success.get(key) != value:
            return None
    if success.get("detail_sha256") != _sha256_file(detail_path):
        return None
    allocations = pd.read_parquet(detail_path)
    if list(allocations.columns) != _detail_columns(contract):
        return None
    if "candidate_hours" in contract and not allocations["candidate_hours"].eq(contract["candidate_hours"]).all():
        return None
    solver_status = pd.DataFrame()
    if contract["allocation_method"] == "optimised_lp":
        status_path = partition / "solver-status.parquet"
        if not status_path.exists() or success.get("solver_status_sha256") != _sha256_file(status_path):
            return None
        solver_status = pd.read_parquet(status_path)
    return allocations, solver_status


# Write a complete partition through a temporary directory with recoverable replacement.
def _write_completed_partition(
    partition: Path,
    allocations: pd.DataFrame,
    solver_status: pd.DataFrame,
    contract: dict,
):
    partition.parent.mkdir(parents=True, exist_ok=True)
    temporary = partition.parent / f".tmp-{partition.name}-{uuid.uuid4().hex}"
    temporary.mkdir(parents=False, exist_ok=False)
    detail_path = temporary / "part-00000.parquet"
    allocations[_detail_columns(contract)].to_parquet(
        detail_path, index=False, compression="zstd"
    )
    success = {
        **contract,
        "detail_row_count": len(allocations),
        "detail_shifted_mwh": float(allocations["shifted_mwh"].sum()),
        "detail_sha256": _sha256_file(detail_path),
    }
    if contract["allocation_method"] == "optimised_lp":
        status_path = temporary / "solver-status.parquet"
        solver_status.to_parquet(status_path, index=False, compression="zstd")
        success["solver_status_sha256"] = _sha256_file(status_path)
    (temporary / "_SUCCESS.json").write_text(json.dumps(success, indent=2, sort_keys=True) + "\n")
    if partition.exists():
        backup = partition.parent / f".backup-{partition.name}-{uuid.uuid4().hex}"
        partition.replace(backup)
        try:
            temporary.replace(partition)
        except BaseException:
            backup.replace(partition)
            raise
        shutil.rmtree(backup)
    else:
        temporary.replace(partition)

## Check allocation availability

Define checks for all four allocation roots and their scenario partitions before reuse. Collect missing files, incompatible identities and invalid checksum metadata into one error report without changing the allocation files.


In [18]:
def _require_matching_output_root(path: Path, expected_manifest: dict):
    manifest_path = path / "_allocation_root_manifest.json"
    if not path.is_dir() or not manifest_path.is_file():
        raise RuntimeError(f"Required allocation root or manifest is missing: {path}")
    try:
        existing = json.loads(manifest_path.read_text())
    except (OSError, json.JSONDecodeError) as exc:
        raise RuntimeError(f"Cannot read allocation root manifest {manifest_path}: {exc}") from exc
    if existing != expected_manifest:
        changed = (
            sorted(key for key in set(existing) | set(expected_manifest)
                   if existing.get(key) != expected_manifest.get(key))
            if isinstance(existing, dict) else ["manifest structure"]
        )
        raise RuntimeError(
            f"Allocation root does not match the requested inputs/settings: {path}. "
            f"Differing fields: {', '.join(changed)}. Existing files were preserved."
        )


def preflight_allocation_caches(
    *, hourly_path: Path, utilisation_path: Path, flexibility_path: Path,
    allocation_roots,
):
    """Check all four cache roots without reading large allocation tables or writing files.

    Detailed checksums, column schemas and scientific row checks still run when
    each partition is loaded by the driver. This preflight detects missing or
    incompatible later partitions before earlier modes write their summaries.
    """
    _, _, matrix = read_scenario_matrix(utilisation_path, flexibility_path)
    roots = list(allocation_roots)
    expected_modes = {
        (hours, mode)
        for hours in (6, 9)
        for mode in ("daily_spatiotemporal", "same_hour_geographic")
    }
    if len(roots) != len(expected_modes) or {(hours, mode) for hours, mode, _ in roots} != expected_modes:
        raise ValueError("Allocation preflight requires both 6h/9h settings and both shift modes exactly once")
    issues = []
    for candidate_hours, shift_mode, root in roots:
        root = Path(root)
        try:
            manifest = _root_manifest(
                hourly_path, utilisation_path, flexibility_path, shift_mode,
                candidate_hours,
            )
            _require_matching_output_root(root, manifest)
        except (OSError, RuntimeError) as exc:
            issues.append(str(exc))
            continue
        for scenario in matrix.to_dict("records"):
            for allocation_method in ALLOCATION_METHODS:
                partition = _partition_path(
                    root, allocation_method, scenario["utilisation_scenario_id"],
                    scenario["flexibility_scenario_id"],
                )
                contract = _partition_contract(allocation_method, scenario, shift_mode, candidate_hours)
                marker_path = partition / "_SUCCESS.json"
                detail_path = partition / "part-00000.parquet"
                if not marker_path.is_file() or not detail_path.is_file():
                    issues.append(f"Missing completed allocation files: {partition}")
                    continue
                try:
                    marker = json.loads(marker_path.read_text())
                except (OSError, json.JSONDecodeError) as exc:
                    issues.append(f"Cannot read allocation completion marker {marker_path}: {exc}")
                    continue
                if not isinstance(marker, dict) or any(marker.get(key) != value for key, value in contract.items()):
                    issues.append(f"Allocation completion identity does not match: {marker_path}")
                    continue
                hashes = ["detail_sha256"]
                if allocation_method == "optimised_lp":
                    hashes.append("solver_status_sha256")
                    if not (partition / "solver-status.parquet").is_file():
                        issues.append(f"Missing optimisation status file: {partition}")
                if any(not isinstance(marker.get(key), str) or len(marker[key]) != 64
                       or any(char not in "0123456789abcdef" for char in marker[key])
                       for key in hashes):
                    issues.append(f"Missing or invalid completion checksum metadata: {marker_path}")
    if issues:
        raise RuntimeError("Cannot reuse the requested allocation caches:\n- " + "\n- ".join(issues))

## Validate detailed transfers

Define checks for allocation method, positive shifted energy and renewable gain, different query locations and consistent weighted metrics. Check exact timestamp equality for same-hour transfers and a shared UTC date for daily transfers.


In [19]:
def _validate_detailed_allocations(allocations: pd.DataFrame, shift_mode: str, allocation_method: str):
    if allocations.empty:
        return
    if set(allocations["allocation_method"]) != {allocation_method}:
        raise ValueError("Detailed partition contains the wrong allocation method")
    if (allocations["shifted_mwh"] <= ALLOCATION_TOLERANCE_MWH).any():
        raise ValueError("Allocation output contains non-positive shifted energy")
    if (allocations["renewable_availability_gain"] <= ALLOCATION_TOLERANCE_MWH).any():
        raise ValueError("Allocation output contains non-positive renewable gain")
    if (allocations["sender_query_location_id"].astype(str) == allocations["receiver_query_location_id"].astype(str)).any():
        raise ValueError("Same-location transfers are present")
    if not np.allclose(
        allocations["weighted_renewable_gain"],
        allocations["shifted_mwh"] * allocations["renewable_availability_gain"],
        atol=1e-8,
        rtol=1e-10,
    ):
        raise ValueError("Weighted renewable-gain contributions are inconsistent")
    if not np.allclose(
        allocations["weighted_abs_time_hours"],
        allocations["shifted_mwh"] * allocations["time_difference_hours"],
        atol=1e-8,
        rtol=1e-10,
    ):
        raise ValueError("Weighted time contributions are inconsistent")
    if shift_mode == "same_hour_geographic":
        if allocations["time_difference_hours"].abs().max() > FEASIBILITY_TOLERANCE_MWH:
            raise ValueError("Same-hour allocations contain non-zero time differences")
        if not (allocations["sender_time_utc"] == allocations["receiver_time_utc"]).all():
            raise ValueError("Same-hour allocations do not preserve exact UTC timestamps")
    elif not (
        pd.to_datetime(allocations["sender_time_utc"], utc=True).dt.date
        == pd.to_datetime(allocations["receiver_time_utc"], utc=True).dt.date
    ).all():
        raise ValueError("Daily allocations cross a UTC-date boundary")

## Validate operator-year summaries

Define checks for both allocation methods, all workload cases, the 85% utilisation and 25% flexibility reference, and reported realised utilisation. Check that shifted energy does not decrease as the flexible workload share increases within the comparison tolerance.


In [20]:
def validate_operator_year_summary(summary):
    """Check method/scenario coverage, reference flags and flexible-share results."""
    if summary["realised_base_utilisation"].isna().any():
        raise ValueError("Realised utilisation is missing from the operator-year summary")
    cases = summary[["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id"]].drop_duplicates()
    if len(cases) != 18:
        raise ValueError(f"Expected 18 allocation-method/scenario combinations, found {len(cases)}")
    if set(summary["allocation_method"]) != set(ALLOCATION_METHODS):
        raise ValueError("Both allocation methods are not present")

    monotonicity = summary.pivot_table(
        index=["allocation_method", "utilisation_scenario_id", "shift_mode", "model_layer", "operator_group", "weather_year"],
        columns="flexibility_scenario_id",
        values="shifted_mwh",
        aggfunc="sum",
    )
    if ((monotonicity["flex_25"] + OBJECTIVE_TOLERANCE < monotonicity["flex_20"]) | (monotonicity["flex_30"] + OBJECTIVE_TOLERANCE < monotonicity["flex_25"])).any():
        raise ValueError("Shifted energy is not non-decreasing with flexible workload share")

    central_cases = summary.loc[
        summary["is_central"],
        ["utilisation_scenario_id", "flexibility_scenario_id"],
    ].drop_duplicates()
    if central_cases.to_dict("records") != [{"utilisation_scenario_id": "util_85", "flexibility_scenario_id": "flex_25"}]:
        raise ValueError("Central summary is not util_85/flex_25")

## Prepare the allocation driver

Define the driver to reuse completed partitions and validate the resulting summaries for one candidate-hour setting and shift mode. Missing or invalid partitions stop execution when new allocations are disabled. When enabled, calculate missing partitions, assemble parallel LP results in order and save the validated results.


In [21]:
# Reuse or calculate each scenario partition and export the validated summary family.
def run_allocation_scenarios(
    *, hourly_path: Path, utilisation_path: Path, flexibility_path: Path,
    shift_mode: str, parquet_output_dir: Path, operator_year_output_path: Path,
    central_daily_output_path: Path, central_operator_year_output_path: Path,
    hourly_base: pd.DataFrame | None = None,
    candidate_hours: int = 6,
    allow_new_allocations: bool,
):
    """Read matching partitions or calculate missing scenarios for either hour setting."""
    if not isinstance(allow_new_allocations, bool):
        raise TypeError("allow_new_allocations must be True or False")
    if candidate_hours not in (6, 9):
        raise ValueError("This comparison supports exactly 6 and 9 candidate hours")
    utilisation, _, matrix = read_scenario_matrix(utilisation_path, flexibility_path)
    if hourly_base is None:
        hourly_base = read_hourly_model_inputs(hourly_path)
    hourly_base = apply_candidate_hours(hourly_base, candidate_hours)
    manifest = _root_manifest(hourly_path, utilisation_path, flexibility_path, shift_mode, candidate_hours)
    if allow_new_allocations:
        _prepare_output_root(parquet_output_dir, manifest)
    else:
        _require_matching_output_root(parquet_output_dir, manifest)
    for output_path in (operator_year_output_path, central_daily_output_path, central_operator_year_output_path):
        output_path.parent.mkdir(parents=True, exist_ok=True)

    all_daily = []
    all_operator_year = []
    all_solver_status = []
    for util in utilisation.to_dict("records"):
        util_id = util["utilisation_scenario_id"]
        hourly = apply_utilisation(hourly_base, util["base_utilisation"])
        grouped_hourly = None
        print(f"{candidate_hours}h {shift_mode} {util_id}")
        util_matrix = matrix[matrix["utilisation_scenario_id"].eq(util_id)]
        for scenario in util_matrix.to_dict("records"):
            flex_id = scenario["flexibility_scenario_id"]
            load_totals = _load_totals(hourly, scenario)
            for allocation_method in ALLOCATION_METHODS:
                partition = _partition_path(parquet_output_dir, allocation_method, util_id, flex_id)
                contract = _partition_contract(allocation_method, scenario, shift_mode, candidate_hours)
                try:
                    cached = _load_completed_partition(partition, contract)
                except Exception as exc:
                    if not allow_new_allocations:
                        raise RuntimeError(f"Cannot reuse allocation partition {partition}: {exc}") from exc
                    raise
                if cached is None and not allow_new_allocations:
                    raise RuntimeError(
                        f"No valid completed allocation partition: {partition}. "
                        "New production allocations are disabled."
                    )
                if cached is not None:
                    allocations, solver_status = cached
                    print(f"  reused {allocation_method} {util_id}/{flex_id}: {len(allocations):,} arcs")
                else:
                    if grouped_hourly is None:
                        grouped_hourly = _group_hourly_for_mode(hourly, shift_mode)
                    send_by_group, receive_by_group, common_keys = grouped_hourly
                    rows = []
                    statuses = []
                    if allocation_method == "greedy":
                        progress_every = 1000 if shift_mode == "daily_spatiotemporal" else 10000
                        for index, key in enumerate(common_keys, 1):
                            rows.extend(
                                _allocate_group_greedy(
                                    key, send_by_group[key], receive_by_group[key], scenario, shift_mode
                                )
                            )
                            if index % progress_every == 0:
                                print(
                                    f"  greedy {util_id}/{flex_id}: "
                                    f"{index:,}/{len(common_keys):,} groups; {len(rows):,} arcs"
                                )
                    else:
                        batch_iterator = enumerate(
                            _iter_optimisation_batches(
                                common_keys,
                                send_by_group,
                                receive_by_group,
                                scenario,
                                shift_mode,
                            ),
                            1,
                        )
                        processed_groups = 0
                        submitted_batches = 0
                        completed_batches = 0
                        next_output_sequence = 1
                        ordered_buffer = {}
                        maximum_in_flight = max(1, OPTIMISATION_MAX_WORKERS * 2)
                        with ThreadPoolExecutor(max_workers=OPTIMISATION_MAX_WORKERS) as executor:
                            pending = {}

                            # Submit the next independent LP batch and record its sequence for ordered results.
                            def submit_next_batch():
                                nonlocal submitted_batches
                                try:
                                    batch_sequence, batch = next(batch_iterator)
                                except StopIteration:
                                    return False
                                future = executor.submit(
                                    _allocate_batch_optimised,
                                    batch,
                                    scenario,
                                    shift_mode,
                                    batch_sequence,
                                )
                                pending[future] = (batch_sequence, len(batch))
                                submitted_batches += 1
                                return True

                            for _ in range(maximum_in_flight):
                                if not submit_next_batch():
                                    break
                            try:
                                while pending:
                                    done, _ = wait(pending, return_when=FIRST_COMPLETED)
                                    for future in done:
                                        batch_sequence, group_count = pending.pop(future)
                                        batch_rows, batch_statuses = future.result()
                                        ordered_buffer[batch_sequence] = (
                                            batch_rows,
                                            batch_statuses,
                                            group_count,
                                        )
                                        completed_batches += 1
                                        submit_next_batch()
                                    while next_output_sequence in ordered_buffer:
                                        batch_rows, batch_statuses, group_count = ordered_buffer.pop(
                                            next_output_sequence
                                        )
                                        rows.extend(batch_rows)
                                        statuses.extend(batch_statuses)
                                        processed_groups += group_count
                                        next_output_sequence += 1
                                    if completed_batches % 100 == 0:
                                        print(
                                            f"  optimised_lp {util_id}/{flex_id}: "
                                            f"{completed_batches:,}/{submitted_batches:,}+ batches solved; "
                                            f"{processed_groups:,}/{len(common_keys):,} groups assembled"
                                        )
                            except BaseException:
                                for future in pending:
                                    future.cancel()
                                raise
                        if ordered_buffer or processed_groups != len(common_keys):
                            raise RuntimeError(
                                "Parallel optimisation did not return every temporal group in order"
                            )
                    allocations = pd.DataFrame(rows, columns=ALLOCATION_OUTPUT_COLUMNS)
                    solver_status = (
                        _aggregate_solver_status(statuses, scenario)
                        if allocation_method == "optimised_lp"
                        else pd.DataFrame()
                    )
                    if candidate_hours == 9:
                        allocations["candidate_hours"] = candidate_hours
                        solver_status["candidate_hours"] = candidate_hours

                _validate_detailed_allocations(allocations, shift_mode, allocation_method)
                if allow_new_allocations and cached is None:
                    _write_completed_partition(partition, allocations, solver_status, contract)
                    print(f"  wrote {allocation_method} {util_id}/{flex_id}: {len(allocations):,} arcs")
                daily, operator_year = summarize_allocations(
                    allocations,
                    load_totals,
                    allocation_method=allocation_method,
                    shift_mode=shift_mode,
                )
                all_daily.append(daily)
                all_operator_year.append(operator_year)
                if allocation_method == "optimised_lp":
                    all_solver_status.append(solver_status)

    daily_summary = pd.concat(all_daily, ignore_index=True)
    operator_year_summary = pd.concat(all_operator_year, ignore_index=True)
    solver_status_summary = pd.concat(all_solver_status, ignore_index=True)
    operator_year_summary = operator_year_summary.sort_values(
        ["allocation_method", "base_utilisation", "flexible_workload_share", "model_layer", "operator_group", "weather_year"]
    ).reset_index(drop=True)

    if candidate_hours == 9:
        for frame in (daily_summary, operator_year_summary, solver_status_summary):
            frame["candidate_hours"] = candidate_hours
    validate_operator_year_summary(operator_year_summary)
    operator_year_summary.to_csv(operator_year_output_path, index=False)

    central_daily = daily_summary[daily_summary["is_central"]].drop(columns=SCENARIO_METADATA_COLUMNS)
    central_operator_year = operator_year_summary[operator_year_summary["is_central"]].drop(columns=SCENARIO_METADATA_COLUMNS)
    central_daily.to_csv(central_daily_output_path, index=False)
    central_operator_year.to_csv(central_operator_year_output_path, index=False)

    return daily_summary, operator_year_summary, matrix, solver_status_summary

## Compare shift modes and allocation methods

Define the combined totals, reference-case tables and matched greedy-versus-LP comparisons for each hour setting. Calculate annual averages separately from sums across weather years and retain the energy-weighted renewable and time metrics.


In [22]:
def build_comparison_tables(comparison):
    """Aggregate one candidate-hour setting and compare matched allocation methods."""
    total_group_cols = SCENARIO_METADATA_COLUMNS + ["allocation_method", "shift_mode", "model_layer"]
    total = (
        comparison.groupby(total_group_cols, dropna=False)
        .agg(
            years_covered=("weather_year", "nunique"),
            operators=("operator_group", "nunique"),
            shifted_mwh=("shifted_mwh", "sum"),
            allocation_count=("allocation_count", "sum"),
            total_weighted_renewable_gain=("total_weighted_renewable_gain", "sum"),
            total_weighted_abs_time_hours=("total_weighted_abs_time_hours", "sum"),
            total_actual_load_mwh=("total_actual_load_mwh", "sum"),
            total_capacity_hours_mwh=("total_capacity_hours_mwh", "sum"),
            capped_hour_count=("capped_hour_count", "sum"),
            hourly_observation_count=("hourly_observation_count", "sum"),
        )
        .reset_index()
    )
    total = _weighted_metrics(total)
    total["realised_base_utilisation"] = total["total_actual_load_mwh"] / total["total_capacity_hours_mwh"]
    total["average_annual_shifted_mwh"] = total["shifted_mwh"] / total["years_covered"]
    ratio = total.pivot_table(
        index=["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "model_layer"],
        columns="shift_mode",
        values="shifted_mwh",
        aggfunc="sum",
    ).reset_index()
    ratio["same_hour_share_of_daily_spatiotemporal"] = (
        ratio["same_hour_geographic"] / ratio["daily_spatiotemporal"]
    )
    total = total.merge(
        ratio[[
            "allocation_method", "utilisation_scenario_id", "flexibility_scenario_id",
            "model_layer", "same_hour_share_of_daily_spatiotemporal",
        ]],
        on=["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "model_layer"],
        how="left",
        validate="many_to_one",
    )
    total = total.sort_values([
        "allocation_method", "base_utilisation", "flexible_workload_share", "shift_mode", "model_layer"
    ]).reset_index(drop=True)


    central_comparison = comparison[comparison["is_central"]].drop(columns=SCENARIO_METADATA_COLUMNS)
    central_total = total[total["is_central"]].drop(columns=SCENARIO_METADATA_COLUMNS)



    gap_keys = SCENARIO_METADATA_COLUMNS + [
        "shift_mode", "model_layer", "operator_group", "weather_year"
    ]
    gap_value_columns = [
        "shifted_mwh", "total_weighted_renewable_gain", "total_weighted_abs_time_hours",
        "mwh_weighted_renewable_availability_gain",
        "mwh_weighted_absolute_time_difference_hours",
    ]
    greedy_gap = comparison[comparison["allocation_method"].eq("greedy")][gap_keys + gap_value_columns]
    optimised_gap = comparison[comparison["allocation_method"].eq("optimised_lp")][gap_keys + gap_value_columns]
    operator_gap = greedy_gap.merge(
        optimised_gap,
        on=gap_keys,
        how="outer",
        validate="one_to_one",
        suffixes=("_greedy", "_optimised_lp"),
    )
    if operator_gap.filter(regex="_(greedy|optimised_lp)$").isna().any().any():
        raise ValueError("Greedy and optimised operator/year rows do not align")
    operator_gap["optimality_gap_mwh"] = operator_gap["shifted_mwh_optimised_lp"] - operator_gap["shifted_mwh_greedy"]
    operator_gap["optimality_gap_percent_of_greedy"] = np.divide(
        operator_gap["optimality_gap_mwh"] * 100,
        operator_gap["shifted_mwh_greedy"],
        out=np.zeros(len(operator_gap), dtype=float),
        where=operator_gap["shifted_mwh_greedy"].to_numpy(dtype=float) > ALLOCATION_TOLERANCE_MWH,
    )


    total_gap_keys = SCENARIO_METADATA_COLUMNS + ["shift_mode", "model_layer"]
    total_gap = (
        operator_gap.groupby(total_gap_keys, dropna=False)
        .agg(
            shifted_mwh_greedy=("shifted_mwh_greedy", "sum"),
            shifted_mwh_optimised_lp=("shifted_mwh_optimised_lp", "sum"),
            total_weighted_renewable_gain_greedy=("total_weighted_renewable_gain_greedy", "sum"),
            total_weighted_renewable_gain_optimised_lp=("total_weighted_renewable_gain_optimised_lp", "sum"),
            total_weighted_abs_time_hours_greedy=("total_weighted_abs_time_hours_greedy", "sum"),
            total_weighted_abs_time_hours_optimised_lp=("total_weighted_abs_time_hours_optimised_lp", "sum"),
        )
        .reset_index()
    )
    for method in ALLOCATION_METHODS:
        total_gap[f"mwh_weighted_renewable_availability_gain_{method}"] = np.divide(
            total_gap[f"total_weighted_renewable_gain_{method}"],
            total_gap[f"shifted_mwh_{method}"],
            out=np.zeros(len(total_gap), dtype=float),
            where=total_gap[f"shifted_mwh_{method}"].to_numpy(dtype=float) > ALLOCATION_TOLERANCE_MWH,
        )
        total_gap[f"mwh_weighted_absolute_time_difference_hours_{method}"] = np.divide(
            total_gap[f"total_weighted_abs_time_hours_{method}"],
            total_gap[f"shifted_mwh_{method}"],
            out=np.zeros(len(total_gap), dtype=float),
            where=total_gap[f"shifted_mwh_{method}"].to_numpy(dtype=float) > ALLOCATION_TOLERANCE_MWH,
        )
    total_gap["optimality_gap_mwh"] = total_gap["shifted_mwh_optimised_lp"] - total_gap["shifted_mwh_greedy"]
    total_gap["optimality_gap_percent_of_greedy"] = np.divide(
        total_gap["optimality_gap_mwh"] * 100,
        total_gap["shifted_mwh_greedy"],
        out=np.zeros(len(total_gap), dtype=float),
        where=total_gap["shifted_mwh_greedy"].to_numpy(dtype=float) > ALLOCATION_TOLERANCE_MWH,
    )

    return total, central_comparison, central_total, operator_gap, total_gap

## Check combined allocation results

Define checks that both shift modes are present and that every feasible LP group completed the required optimisation stages. Verify objective preservation and compare LP and greedy energy using the accumulated solver tolerances.


In [23]:
def validate_comparison_tables(comparison, solver_status, operator_gap):
    """Check both shift modes and accumulated solver tolerances after summary validation."""
    gap_keys = SCENARIO_METADATA_COLUMNS + ["shift_mode", "model_layer", "operator_group", "weather_year"]
    if set(comparison["shift_mode"]) != {"daily_spatiotemporal", "same_hour_geographic"}:
        raise ValueError("Both shift modes are not present")

    # Stage 2/3 preserve the stage-1 optimum within a declared tolerance for each
    # feasible temporal group.  When operator-year totals sum hundreds of groups,
    # those permitted sub-Wh differences can exceed one global scalar tolerance.
    # Derive a conservative aggregate tolerance from the recorded solver contract.
    operator_gap_tolerance = solver_status[gap_keys + [
        "feasible_edge_group_count",
        "stage1_optimal_shifted_mwh",
        "maximum_stage1_preservation_tolerance_mwh",
    ]].copy()
    operator_gap_tolerance["aggregate_stage1_tolerance_mwh"] = (
        operator_gap_tolerance["feasible_edge_group_count"]
        * (
            operator_gap_tolerance["maximum_stage1_preservation_tolerance_mwh"]
            + FEASIBILITY_TOLERANCE_MWH
        )
    )
    operator_gap_check = operator_gap.merge(
        operator_gap_tolerance,
        on=gap_keys,
        how="left",
        validate="one_to_one",
    )
    if operator_gap_check[["stage1_optimal_shifted_mwh", "aggregate_stage1_tolerance_mwh"]].isna().any().any():
        raise ValueError("Solver tolerance metadata is missing from the optimiser-versus-greedy check")
    operator_gap_check["stage1_minus_greedy_mwh"] = (
        operator_gap_check["stage1_optimal_shifted_mwh"]
        - operator_gap_check["shifted_mwh_greedy"]
    )
    stage1_failure = operator_gap_check["stage1_minus_greedy_mwh"] < -operator_gap_check["aggregate_stage1_tolerance_mwh"]
    if stage1_failure.any():
        worst = operator_gap_check.loc[
            operator_gap_check["stage1_minus_greedy_mwh"].idxmin()
        ]
        raise ValueError(
            "Greedy shifted MWh exceeds the stage-1 optimum beyond accumulated "
            f"solver tolerance for {worst[gap_keys].to_dict()}"
        )
    final_gap_failure = operator_gap_check["optimality_gap_mwh"] < -operator_gap_check["aggregate_stage1_tolerance_mwh"]
    if final_gap_failure.any():
        worst = operator_gap_check.loc[operator_gap_check["optimality_gap_mwh"].idxmin()]
        raise ValueError(
            "Final optimised shifted MWh is below greedy beyond accumulated "
            f"solver tolerance for {worst[gap_keys].to_dict()}"
        )

    if not (solver_status["stage1_optimal_group_count"] == solver_status["feasible_edge_group_count"]).all():
        raise ValueError("Not every feasible group reached an optimal stage-1 status")
    if not (solver_status["stage2_optimal_group_count"] == solver_status["feasible_edge_group_count"]).all():
        raise ValueError("Not every feasible group reached an optimal stage-2 status")
    daily_status = solver_status[solver_status["shift_mode"].eq("daily_spatiotemporal")]
    same_hour_status = solver_status[solver_status["shift_mode"].eq("same_hour_geographic")]
    if not (daily_status["stage3_optimal_group_count"] == daily_status["feasible_edge_group_count"]).all():
        raise ValueError("Not every feasible daily group reached an optimal stage-3 status")
    if not (same_hour_status["stage3_not_applicable_group_count"] == same_hour_status["feasible_edge_group_count"]).all():
        raise ValueError("Same-hour groups unexpectedly ran stage 3")

    if (
        solver_status["maximum_absolute_stage1_preservation_error_mwh"]
        > solver_status["maximum_stage1_preservation_tolerance_mwh"] + FEASIBILITY_TOLERANCE_MWH
    ).any():
        raise ValueError("Final optimisation flows do not preserve stage-1 shifted MWh within tolerance")
    if (
        solver_status["maximum_absolute_stage2_preservation_error_weighted_gain"]
        > solver_status["maximum_stage2_preservation_tolerance_weighted_gain"] + FEASIBILITY_TOLERANCE_MWH
    ).any():
        raise ValueError("Final optimisation flows do not preserve stage-2 gain within tolerance")

## Define synthetic allocation examples

Prepare small test networks for splitting transfers, limited headroom, no feasible edges and a case where greedy shifts less energy than LP. Test the daily time objective and the omission of that stage for same-hour shifting.


In [24]:
# Run one synthetic temporal group through the shared batched LP allocator.
def _allocate_group_optimised(key, sends, receives, scenario, shift_mode: str):
    """Single-group wrapper used only by focused synthetic tests."""
    group = {
        "key": key,
        "sends": sends,
        "receives": receives,
        "network": _feasible_network(
            sends,
            receives,
            float(scenario["flexible_workload_share"]),
        ),
    }
    allocations, statuses = _allocate_batch_optimised(
        [group], scenario, shift_mode, batch_sequence=1
    )
    if len(statuses) != 1:
        raise RuntimeError("Single-group optimisation did not return one status record")
    return allocations, statuses[0]


# Construct a small hourly test frame with the required geographic and clock fields.
def _synthetic_frame(rows):
    frame = pd.DataFrame(rows)
    defaults = {
        "country": "Testland", "country_code": "TT", "timezone": "UTC",
        "renewable_query_region": "Test region", "utc_hour": 0,
        "eligible_receive_mwh": 0.0, "actual_total_load_mw": 0.0,
    }
    for column, value in defaults.items():
        if column not in frame:
            frame[column] = value
        else:
            frame[column] = frame[column].fillna(value)
    frame["time_utc"] = pd.to_datetime(frame["time_utc"], utc=True)
    frame["utc_date"] = frame["time_utc"].dt.date
    frame["utc_hour"] = frame["time_utc"].dt.hour
    return frame


# Define a fully flexible synthetic workload for the allocator examples.
def _synthetic_scenario():
    return {
        "utilisation_scenario_id": "synthetic_util",
        "base_utilisation": 1.0,
        "utilisation_scenario_label": "Synthetic",
        "is_central_utilisation": False,
        "flexibility_scenario_id": "synthetic_flex",
        "flexible_workload_share": 1.0,
        "scenario_label": "Synthetic",
        "is_central_flexibility": False,
        "is_central": False,
    }


# Check controlled examples of splitting, headroom, zero edges and LP objectives.
def run_synthetic_allocator_tests():
    """Exercise splitting, binding headroom, zero edges, and greedy suboptimality."""
    scenario = _synthetic_scenario()
    daily_key = ("synthetic", "Synthetic operator", 2025, pd.Timestamp("2025-01-01").date())

    sends = _synthetic_frame([
        {"query_location_id": "sender", "time_utc": "2025-01-01T00:00:00Z", "renewable_availability_index": 0.0, "actual_total_load_mw": 10.0}
    ])
    receives = _synthetic_frame([
        {"query_location_id": "receiver_a", "time_utc": "2025-01-01T01:00:00Z", "renewable_availability_index": 0.5, "eligible_receive_mwh": 5.0},
        {"query_location_id": "receiver_b", "time_utc": "2025-01-01T02:00:00Z", "renewable_availability_index": 0.4, "eligible_receive_mwh": 3.0},
    ])
    split = _allocate_group_greedy(daily_key, sends, receives, scenario, "daily_spatiotemporal")
    if len(split) != 2 or not np.isclose(sum(row["shifted_mwh"] for row in split), 8.0):
        raise AssertionError("Synthetic split/headroom/unused-potential test failed")

    zero_receivers = receives.copy()
    zero_receivers["query_location_id"] = "sender"
    zero_receivers["renewable_availability_index"] = -1.0
    zero_greedy = _allocate_group_greedy(daily_key, sends, zero_receivers, scenario, "daily_spatiotemporal")
    zero_lp, zero_status = _allocate_group_optimised(daily_key, sends, zero_receivers, scenario, "daily_spatiotemporal")
    if zero_greedy or zero_lp or zero_status["feasible_edge_count"] != 0:
        raise AssertionError("Synthetic zero-edge test failed")

    suboptimal_sends = _synthetic_frame([
        {"query_location_id": "sender_1", "time_utc": "2025-01-01T00:00:00Z", "renewable_availability_index": 0.0, "actual_total_load_mw": 1.0},
        {"query_location_id": "sender_2", "time_utc": "2025-01-01T00:00:00Z", "renewable_availability_index": 0.2, "actual_total_load_mw": 1.0},
    ])
    suboptimal_receivers = _synthetic_frame([
        {"query_location_id": "receiver_1", "time_utc": "2025-01-01T01:00:00Z", "renewable_availability_index": 0.3, "eligible_receive_mwh": 1.0},
        {"query_location_id": "receiver_2", "time_utc": "2025-01-01T01:00:00Z", "renewable_availability_index": 0.1, "eligible_receive_mwh": 1.0},
    ])
    greedy = _allocate_group_greedy(daily_key, suboptimal_sends, suboptimal_receivers, scenario, "daily_spatiotemporal")
    optimised, status = _allocate_group_optimised(daily_key, suboptimal_sends, suboptimal_receivers, scenario, "daily_spatiotemporal")
    greedy_total = sum(row["shifted_mwh"] for row in greedy)
    optimised_total = sum(row["shifted_mwh"] for row in optimised)
    if not np.isclose(greedy_total, 1.0) or not np.isclose(optimised_total, 2.0):
        raise AssertionError("Synthetic deliberately suboptimal greedy case failed")
    if {status["stage1_status"], status["stage2_status"], status["stage3_status"]} != {"optimal"}:
        raise AssertionError("Synthetic daily optimisation stages were not optimal")

    closest_receivers = _synthetic_frame([
        {"query_location_id": "far", "time_utc": "2025-01-01T23:00:00Z", "renewable_availability_index": 0.5, "eligible_receive_mwh": 1.0},
        {"query_location_id": "near", "time_utc": "2025-01-01T02:00:00Z", "renewable_availability_index": 0.5, "eligible_receive_mwh": 1.0},
    ])
    closest, _ = _allocate_group_optimised(daily_key, sends.assign(actual_total_load_mw=1.0), closest_receivers, scenario, "daily_spatiotemporal")
    if len(closest) != 1 or closest[0]["receiver_query_location_id"] != "near":
        raise AssertionError("Synthetic stage-three time minimisation test failed")

    same_hour_key = ("synthetic", "Synthetic operator", 2025, pd.Timestamp("2025-01-01T00:00:00Z"))
    same_hour_receivers = closest_receivers.copy()
    same_hour_receivers["time_utc"] = pd.Timestamp("2025-01-01T00:00:00Z")
    same_hour_receivers["utc_date"] = pd.Timestamp("2025-01-01").date()
    same_hour, same_status = _allocate_group_optimised(
        same_hour_key, sends.assign(actual_total_load_mw=1.0), same_hour_receivers, scenario, "same_hour_geographic"
    )
    if not same_hour or same_status["stage3_status"] != "not_applicable":
        raise AssertionError("Synthetic same-hour stage test failed")

## Run synthetic allocation checks

Run the synthetic examples to check the allocation constraints and LP objective sequence. Any failed expectation stops execution.


In [25]:
run_synthetic_allocator_tests()

## Check saved allocation partitions

Check all four allocation roots and their scenario partitions before loading the hourly input when new allocations are disabled. Verify input identities, completion markers, required files and checksum metadata, with payload checks performed when each partition is read.


In [26]:
if not ALLOW_NEW_ALLOCATIONS:
    preflight_allocation_caches(
        hourly_path=HOURLY_INPUT_PATH,
        utilisation_path=UTILISATION_SCENARIOS_PATH,
        flexibility_path=FLEXIBILITY_SCENARIOS_PATH,
        allocation_roots=allocation_roots,
    )

## Load shared hourly inputs and scenarios

Read the nine-case workload matrix and the hourly input shared by both allocation methods and hour settings. Validate complete UTC-day rankings and display the scenario definitions and hourly row count.


In [27]:
_, _, scenario_matrix = read_scenario_matrix(UTILISATION_SCENARIOS_PATH, FLEXIBILITY_SCENARIOS_PATH)
print("Shared workload scenario matrix (utilisation and flexibility are fractions):")
print(scenario_matrix[[
    "utilisation_scenario_id", "base_utilisation", "flexibility_scenario_id",
    "flexible_workload_share", "is_central"
]].to_string(index=False))

hourly_base = read_hourly_model_inputs(HOURLY_INPUT_PATH)
validate_candidate_hour_counts(hourly_base)

print(f"Hourly model rows: {len(hourly_base):,}")

Shared workload scenario matrix (utilisation and flexibility are fractions):
utilisation_scenario_id  base_utilisation flexibility_scenario_id  flexible_workload_share  is_central
                util_80              0.80                 flex_20                     0.20       False
                util_80              0.80                 flex_25                     0.25       False
                util_80              0.80                 flex_30                     0.30       False
                util_85              0.85                 flex_20                     0.20       False
                util_85              0.85                 flex_25                     0.25        True
                util_85              0.85                 flex_30                     0.30       False
                util_90              0.90                 flex_20                     0.20       False
                util_90              0.90                 flex_25                     0.25       Fa

## Six-hour daily shifting

Reuse the six-hour daily allocations for all workload cases and both methods, allowing transfers between different hours within one UTC date. Display shifted GWh summed across operators and weather years for each method, workload case and model layer.


In [28]:
_, daily_operator_year_6h, _, daily_solver_status_6h = run_allocation_scenarios(
    hourly_path=HOURLY_INPUT_PATH,
    hourly_base=hourly_base,
    utilisation_path=UTILISATION_SCENARIOS_PATH,
    flexibility_path=FLEXIBILITY_SCENARIOS_PATH,
    shift_mode="daily_spatiotemporal",
    parquet_output_dir=DAILY_PARQUET_OUTPUT_DIR,
    operator_year_output_path=DAILY_OPERATOR_YEAR_PATH,
    central_daily_output_path=DAILY_CENTRAL_DAILY_PATH,
    central_operator_year_output_path=DAILY_CENTRAL_OPERATOR_YEAR_PATH,
    candidate_hours=SIX_HOUR_CANDIDATE_HOURS,
    allow_new_allocations=ALLOW_NEW_ALLOCATIONS,
)

daily_preview_6h = (
    daily_operator_year_6h.groupby(
        ["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "model_layer"],
        as_index=False,
    )["shifted_mwh"].sum()
    .assign(shifted_gwh_all_weather_years=lambda frame: frame["shifted_mwh"] / 1000)
    .drop(columns="shifted_mwh")
)

print("6h daily spatio-temporal mode: total shifted GWh across operators and all configured weather years:")
print(daily_preview_6h.to_string(index=False))

6h daily_spatiotemporal util_80
  reused greedy util_80/flex_20: 2,196,911 arcs
  reused optimised_lp util_80/flex_20: 2,198,211 arcs
  reused greedy util_80/flex_25: 2,043,668 arcs
  reused optimised_lp util_80/flex_25: 2,059,052 arcs
  reused greedy util_80/flex_30: 1,914,745 arcs
  reused optimised_lp util_80/flex_30: 1,947,422 arcs
6h daily_spatiotemporal util_85
  reused greedy util_85/flex_20: 1,818,063 arcs
  reused optimised_lp util_85/flex_20: 1,864,872 arcs
  reused greedy util_85/flex_25: 1,702,189 arcs
  reused optimised_lp util_85/flex_25: 1,765,036 arcs
  reused greedy util_85/flex_30: 1,623,277 arcs
  reused optimised_lp util_85/flex_30: 1,696,214 arcs
6h daily_spatiotemporal util_90
  reused greedy util_90/flex_20: 1,184,977 arcs
  reused optimised_lp util_90/flex_20: 1,242,585 arcs
  reused greedy util_90/flex_25: 1,143,250 arcs
  reused optimised_lp util_90/flex_25: 1,199,384 arcs
  reused greedy util_90/flex_30: 1,115,909 arcs
  reused optimised_lp util_90/flex_30: 1

## Six-hour same-hour shifting

Reuse the six-hour geographic allocations, where transfers share an exact UTC timestamp. Display shifted GWh across operators and weather years for each method, workload case and model layer.


In [29]:
_, same_hour_operator_year_6h, _, same_hour_solver_status_6h = run_allocation_scenarios(
    hourly_path=HOURLY_INPUT_PATH,
    hourly_base=hourly_base,
    utilisation_path=UTILISATION_SCENARIOS_PATH,
    flexibility_path=FLEXIBILITY_SCENARIOS_PATH,
    shift_mode="same_hour_geographic",
    parquet_output_dir=SAME_HOUR_PARQUET_OUTPUT_DIR,
    operator_year_output_path=SAME_HOUR_OPERATOR_YEAR_PATH,
    central_daily_output_path=SAME_HOUR_CENTRAL_DAILY_PATH,
    central_operator_year_output_path=SAME_HOUR_CENTRAL_OPERATOR_YEAR_PATH,
    candidate_hours=SIX_HOUR_CANDIDATE_HOURS,
    allow_new_allocations=ALLOW_NEW_ALLOCATIONS,
)

same_hour_preview_6h = (
    same_hour_operator_year_6h.groupby(
        ["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "model_layer"],
        as_index=False,
    )["shifted_mwh"].sum()
    .assign(shifted_gwh_all_weather_years=lambda frame: frame["shifted_mwh"] / 1000)
    .drop(columns="shifted_mwh")
)

print("6h same-hour geographic mode: total shifted GWh across operators and all configured weather years:")
print(same_hour_preview_6h.to_string(index=False))

6h same_hour_geographic util_80
  reused greedy util_80/flex_20: 1,321,310 arcs
  reused optimised_lp util_80/flex_20: 1,351,096 arcs
  reused greedy util_80/flex_25: 1,313,854 arcs
  reused optimised_lp util_80/flex_25: 1,340,721 arcs
  reused greedy util_80/flex_30: 1,309,038 arcs
  reused optimised_lp util_80/flex_30: 1,334,306 arcs
6h same_hour_geographic util_85
  reused greedy util_85/flex_20: 1,310,976 arcs
  reused optimised_lp util_85/flex_20: 1,335,704 arcs
  reused greedy util_85/flex_25: 1,305,130 arcs
  reused optimised_lp util_85/flex_25: 1,327,560 arcs
  reused greedy util_85/flex_30: 1,300,971 arcs
  reused optimised_lp util_85/flex_30: 1,321,783 arcs
6h same_hour_geographic util_90
  reused greedy util_90/flex_20: 1,028,992 arcs
  reused optimised_lp util_90/flex_20: 1,042,200 arcs
  reused greedy util_90/flex_25: 1,023,009 arcs
  reused optimised_lp util_90/flex_25: 1,034,527 arcs
  reused greedy util_90/flex_30: 1,018,427 arcs
  reused optimised_lp util_90/flex_30: 1

## Compare and export six-hour results

Combine both six-hour shift modes and prepare totals, reference cases and matched greedy-versus-LP differences. Validate and save the comparison tables, then display annual shifted TWh averaged across the weather years.


In [30]:
comparison_6h = pd.concat([daily_operator_year_6h, same_hour_operator_year_6h], ignore_index=True, sort=False)
comparison_6h = comparison_6h.sort_values([
    "allocation_method", "base_utilisation", "flexible_workload_share", "shift_mode",
    "model_layer", "operator_group", "weather_year",
]).reset_index(drop=True)

solver_status_6h = pd.concat([daily_solver_status_6h, same_hour_solver_status_6h], ignore_index=True)
solver_status_6h = solver_status_6h.sort_values([
    "shift_mode", "base_utilisation", "flexible_workload_share", "model_layer",
    "operator_group", "weather_year",
]).reset_index(drop=True)
total_6h, central_comparison_6h, central_total_6h, operator_gap_6h, total_gap_6h = build_comparison_tables(comparison_6h)
validate_comparison_tables(comparison_6h, solver_status_6h, operator_gap_6h)
comparison_6h.to_csv(COMPARISON_OPERATOR_YEAR_PATH, index=False)
solver_status_6h.to_csv(OPTIMISATION_SOLVER_STATUS_PATH, index=False)
total_6h.to_csv(COMPARISON_TOTAL_PATH, index=False)
central_comparison_6h.to_csv(CENTRAL_COMPARISON_OPERATOR_YEAR_PATH, index=False)
central_total_6h.to_csv(CENTRAL_COMPARISON_TOTAL_PATH, index=False)
operator_gap_6h.to_csv(OPTIMALITY_GAP_OPERATOR_YEAR_PATH, index=False)
total_gap_6h.to_csv(OPTIMALITY_GAP_TOTAL_PATH, index=False)

comparison_preview_6h = total_6h.assign(
    average_annual_shifted_twh=total_6h["average_annual_shifted_mwh"] / 1e6,
)[[
    "allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "shift_mode",
    "model_layer", "average_annual_shifted_twh",
]]

print("6h comparison: average annual shifted TWh (operators summed; mean across weather years):")
print(comparison_preview_6h.to_string(index=False))

6h comparison: average annual shifted TWh (operators summed; mean across weather years):
allocation_method utilisation_scenario_id flexibility_scenario_id           shift_mode         model_layer  average_annual_shifted_twh
           greedy                 util_80                 flex_20 daily_spatiotemporal        current_only                    5.381305
           greedy                 util_80                 flex_20 daily_spatiotemporal current_plus_future                    6.717948
           greedy                 util_80                 flex_20 same_hour_geographic        current_only                    2.200397
           greedy                 util_80                 flex_20 same_hour_geographic current_plus_future                    3.071944
           greedy                 util_80                 flex_25 daily_spatiotemporal        current_only                    5.505446
           greedy                 util_80                 flex_25 daily_spatiotemporal current_plus_f

## Nine-hour daily shifting

Reuse the nine-hour daily allocations for all workload cases and both methods, allowing transfers between different hours within one UTC date. Display shifted GWh summed across operators and weather years for each method, workload case and model layer.


In [31]:
_, daily_operator_year_9h, _, daily_solver_status_9h = run_allocation_scenarios(
    hourly_path=HOURLY_INPUT_PATH,
    hourly_base=hourly_base,
    utilisation_path=UTILISATION_SCENARIOS_PATH,
    flexibility_path=FLEXIBILITY_SCENARIOS_PATH,
    shift_mode="daily_spatiotemporal",
    parquet_output_dir=DAILY_9H_PARQUET_OUTPUT_DIR,
    operator_year_output_path=NINE_HOUR_FINAL_DIR / DAILY_OPERATOR_YEAR_PATH.name,
    central_daily_output_path=NINE_HOUR_FINAL_DIR / DAILY_CENTRAL_DAILY_PATH.name,
    central_operator_year_output_path=NINE_HOUR_FINAL_DIR / DAILY_CENTRAL_OPERATOR_YEAR_PATH.name,
    candidate_hours=NINE_HOUR_CANDIDATE_HOURS,
    allow_new_allocations=ALLOW_NEW_ALLOCATIONS,
)

daily_preview_9h = (
    daily_operator_year_9h.groupby(
        ["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "model_layer"],
        as_index=False,
    )["shifted_mwh"].sum()
    .assign(shifted_gwh_all_weather_years=lambda frame: frame["shifted_mwh"] / 1000)
    .drop(columns="shifted_mwh")
)

print("9h daily spatio-temporal mode: total shifted GWh across operators and all configured weather years:")
print(daily_preview_9h.to_string(index=False))

9h daily_spatiotemporal util_80
  reused greedy util_80/flex_20: 3,311,635 arcs
  reused optimised_lp util_80/flex_20: 3,328,317 arcs
  reused greedy util_80/flex_25: 3,093,608 arcs
  reused optimised_lp util_80/flex_25: 3,116,241 arcs
  reused greedy util_80/flex_30: 2,897,053 arcs
  reused optimised_lp util_80/flex_30: 2,944,471 arcs
9h daily_spatiotemporal util_85
  reused greedy util_85/flex_20: 2,760,826 arcs
  reused optimised_lp util_85/flex_20: 2,829,377 arcs
  reused greedy util_85/flex_25: 2,581,154 arcs
  reused optimised_lp util_85/flex_25: 2,673,793 arcs
  reused greedy util_85/flex_30: 2,459,694 arcs
  reused optimised_lp util_85/flex_30: 2,566,701 arcs
9h daily_spatiotemporal util_90
  reused greedy util_90/flex_20: 1,852,266 arcs
  reused optimised_lp util_90/flex_20: 1,947,636 arcs
  reused greedy util_90/flex_25: 1,782,443 arcs
  reused optimised_lp util_90/flex_25: 1,876,600 arcs
  reused greedy util_90/flex_30: 1,735,023 arcs
  reused optimised_lp util_90/flex_30: 1

## Nine-hour same-hour shifting

Reuse the nine-hour geographic allocations, where transfers share an exact UTC timestamp. Display shifted GWh across operators and weather years for each method, workload case and model layer.


In [32]:
_, same_hour_operator_year_9h, _, same_hour_solver_status_9h = run_allocation_scenarios(
    hourly_path=HOURLY_INPUT_PATH,
    hourly_base=hourly_base,
    utilisation_path=UTILISATION_SCENARIOS_PATH,
    flexibility_path=FLEXIBILITY_SCENARIOS_PATH,
    shift_mode="same_hour_geographic",
    parquet_output_dir=SAME_HOUR_9H_PARQUET_OUTPUT_DIR,
    operator_year_output_path=NINE_HOUR_FINAL_DIR / SAME_HOUR_OPERATOR_YEAR_PATH.name,
    central_daily_output_path=NINE_HOUR_FINAL_DIR / SAME_HOUR_CENTRAL_DAILY_PATH.name,
    central_operator_year_output_path=NINE_HOUR_FINAL_DIR / SAME_HOUR_CENTRAL_OPERATOR_YEAR_PATH.name,
    candidate_hours=NINE_HOUR_CANDIDATE_HOURS,
    allow_new_allocations=ALLOW_NEW_ALLOCATIONS,
)

same_hour_preview_9h = (
    same_hour_operator_year_9h.groupby(
        ["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "model_layer"],
        as_index=False,
    )["shifted_mwh"].sum()
    .assign(shifted_gwh_all_weather_years=lambda frame: frame["shifted_mwh"] / 1000)
    .drop(columns="shifted_mwh")
)

print("9h same-hour geographic mode: total shifted GWh across operators and all configured weather years:")
print(same_hour_preview_9h.to_string(index=False))

9h same_hour_geographic util_80
  reused greedy util_80/flex_20: 2,122,303 arcs
  reused optimised_lp util_80/flex_20: 2,228,858 arcs
  reused greedy util_80/flex_25: 2,117,824 arcs
  reused optimised_lp util_80/flex_25: 2,217,508 arcs
  reused greedy util_80/flex_30: 2,113,950 arcs
  reused optimised_lp util_80/flex_30: 2,208,576 arcs
9h same_hour_geographic util_85
  reused greedy util_85/flex_20: 2,120,695 arcs
  reused optimised_lp util_85/flex_20: 2,211,209 arcs
  reused greedy util_85/flex_25: 2,112,706 arcs
  reused optimised_lp util_85/flex_25: 2,196,040 arcs
  reused greedy util_85/flex_30: 2,105,966 arcs
  reused optimised_lp util_85/flex_30: 2,182,791 arcs
9h same_hour_geographic util_90
  reused greedy util_90/flex_20: 1,711,798 arcs
  reused optimised_lp util_90/flex_20: 1,759,481 arcs
  reused greedy util_90/flex_25: 1,695,298 arcs
  reused optimised_lp util_90/flex_25: 1,737,366 arcs
  reused greedy util_90/flex_30: 1,681,247 arcs
  reused optimised_lp util_90/flex_30: 1

## Compare and export nine-hour results

Combine both nine-hour shift modes and prepare totals, reference cases and matched greedy-versus-LP differences. Validate and save these tables with the candidate-hour setting, then display annual shifted TWh averaged across the weather years.


In [33]:
# Retain the existing daily-then-same-hour row order in nine-hour exports.
comparison_9h = pd.concat([daily_operator_year_9h, same_hour_operator_year_9h], ignore_index=True)
solver_status_9h = pd.concat([daily_solver_status_9h, same_hour_solver_status_9h], ignore_index=True)
total_9h, central_comparison_9h, central_total_9h, operator_gap_9h, total_gap_9h = build_comparison_tables(comparison_9h)
validate_comparison_tables(comparison_9h, solver_status_9h, operator_gap_9h)

for frame, summary_path in (
    (comparison_9h, COMPARISON_OPERATOR_YEAR_PATH), (total_9h, COMPARISON_TOTAL_PATH),
    (central_comparison_9h, CENTRAL_COMPARISON_OPERATOR_YEAR_PATH),
    (central_total_9h, CENTRAL_COMPARISON_TOTAL_PATH), (solver_status_9h, OPTIMISATION_SOLVER_STATUS_PATH),
    (operator_gap_9h, OPTIMALITY_GAP_OPERATOR_YEAR_PATH), (total_gap_9h, OPTIMALITY_GAP_TOTAL_PATH),
):
    frame["candidate_hours"] = NINE_HOUR_CANDIDATE_HOURS
    frame.to_csv(NINE_HOUR_FINAL_DIR / summary_path.name, index=False)
comparison_preview_9h = total_9h.assign(
    average_annual_shifted_twh=total_9h["average_annual_shifted_mwh"] / 1e6,
)[[
    "allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "shift_mode",
    "model_layer", "average_annual_shifted_twh",
]]

print("9h comparison: average annual shifted TWh (operators summed; mean across weather years):")
print(comparison_preview_9h.to_string(index=False))

9h comparison: average annual shifted TWh (operators summed; mean across weather years):
allocation_method utilisation_scenario_id flexibility_scenario_id           shift_mode         model_layer  average_annual_shifted_twh
           greedy                 util_80                 flex_20 daily_spatiotemporal        current_only                    8.175587
           greedy                 util_80                 flex_20 daily_spatiotemporal current_plus_future                   10.216150
           greedy                 util_80                 flex_20 same_hour_geographic        current_only                    3.986821
           greedy                 util_80                 flex_20 same_hour_geographic current_plus_future                    5.414849
           greedy                 util_80                 flex_25 daily_spatiotemporal        current_only                    8.399642
           greedy                 util_80                 flex_25 daily_spatiotemporal current_plus_f

## Combine six-hour and nine-hour results

Define a function to pair matching six-hour and nine-hour results and calculate their differences. Combine both settings into labelled tables and check complete, unique case identities before sorting them for export.


In [34]:
def paired_candidate_uplift(six, nine, keys, volume_column):
    metrics = ["shifted_mwh", "mwh_weighted_renewable_availability_gain",
               "mwh_weighted_absolute_time_difference_hours"]
    if volume_column not in metrics:
        metrics.append(volume_column)
    paired = six[keys + metrics].merge(
        nine[keys + metrics], on=keys, how="outer", validate="one_to_one",
        suffixes=("_6h", "_9h"), indicator=True,
    )
    if not paired["_merge"].eq("both").all():
        raise ValueError("Six-hour and nine-hour comparison keys do not align")
    paired = paired.drop(columns="_merge")
    paired["additional_shifted_mwh"] = paired["shifted_mwh_9h"] - paired["shifted_mwh_6h"]
    paired["shifted_twh_6h"] = paired[f"{volume_column}_6h"] / 1e6
    paired["shifted_twh_9h"] = paired[f"{volume_column}_9h"] / 1e6
    paired["additional_shifted_twh"] = paired["shifted_twh_9h"] - paired["shifted_twh_6h"]
    denominator = paired[f"{volume_column}_6h"].where(paired[f"{volume_column}_6h"].gt(ALLOCATION_TOLERANCE_MWH))
    paired["increase_percent_vs_6h"] = 100 * (paired[f"{volume_column}_9h"] - paired[f"{volume_column}_6h"]) / denominator
    paired["energy_period"] = "weather_year" if volume_column == "shifted_mwh" else "average_annual"
    return paired


candidate_comparison = pd.concat([
    comparison_6h.assign(candidate_hours=6), comparison_9h.assign(candidate_hours=9),
], ignore_index=True)
candidate_total = pd.concat([
    total_6h.assign(candidate_hours=6), total_9h.assign(candidate_hours=9),
], ignore_index=True)
operator_keys = ["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id",
                 "shift_mode", "model_layer", "operator_group", "weather_year"]
total_keys = operator_keys[:-2]
expected_operator_keys = pd.MultiIndex.from_product(
    [ALLOCATION_METHODS, scenario_matrix["utilisation_scenario_id"].unique(),
     scenario_matrix["flexibility_scenario_id"].unique(),
     ["daily_spatiotemporal", "same_hour_geographic"],
     hourly_base["model_layer"].unique(), hourly_base["operator_group"].unique(),
     hourly_base["weather_year"].unique()], names=operator_keys,
)

for hours, frame in candidate_comparison.groupby("candidate_hours"):
    actual_keys = pd.MultiIndex.from_frame(frame[operator_keys])
    if actual_keys.has_duplicates or set(actual_keys) != set(expected_operator_keys):
        raise ValueError(f"Incomplete or duplicate full-matrix operator/year keys for {hours}h")
for frame, keys in ((candidate_comparison, operator_keys), (candidate_total, total_keys)):
    if frame.duplicated(["candidate_hours"] + keys).any():
        raise ValueError("Duplicate candidate-hour comparison keys")
    frame.sort_values(["candidate_hours"] + keys, inplace=True, ignore_index=True)

## Calculate differences between hour settings

Pair matching allocation methods, workload cases, shift modes and model layers to calculate additional shifted energy and percentage changes. Retain individual weather years in the operator comparisons and annual averages in the total comparisons, leaving percentages undefined when the six-hour denominator is zero.


In [35]:
operator_uplift = paired_candidate_uplift(comparison_6h, comparison_9h, operator_keys, "shifted_mwh")
total_uplift = paired_candidate_uplift(total_6h, total_9h, total_keys, "average_annual_shifted_mwh")

## Check the expanded LP network

Check that the nine-hour LP result does not fall below its matching six-hour result beyond the accumulated solver tolerances. Use the recorded objective-preservation and feasibility allowances from both runs because the nine-hour network contains every six-hour candidate.


In [36]:
# Compare achieved LP volumes using accumulated tolerances from BOTH runs.
tolerance_keys = operator_keys[1:]
tolerance = None
for hours, statuses in ((6, solver_status_6h), (9, solver_status_9h)):
    side = statuses[tolerance_keys].copy()
    side[f"tolerance_{hours}h_mwh"] = statuses["feasible_edge_group_count"] * (
        statuses["maximum_stage1_preservation_tolerance_mwh"] + FEASIBILITY_TOLERANCE_MWH
    )
    tolerance = side if tolerance is None else tolerance.merge(side, on=tolerance_keys, validate="one_to_one")
lp_uplift = operator_uplift[operator_uplift["allocation_method"].eq("optimised_lp")].merge(
    tolerance, on=tolerance_keys, how="left", validate="one_to_one",
)
if lp_uplift[["tolerance_6h_mwh", "tolerance_9h_mwh"]].isna().any().any():
    raise ValueError("Missing solver tolerances for the candidate-hour comparison")
if (lp_uplift["additional_shifted_mwh"] < -(lp_uplift["tolerance_6h_mwh"] + lp_uplift["tolerance_9h_mwh"])).any():
    raise ValueError("Nine-hour optimised volume is below six hours beyond accumulated solver tolerance")

## Export the candidate-hour comparison

Save the combined results and paired differences as operator-year and total tables. Display annual shifted TWh and percentage changes for each matched case.


In [37]:
for frame, path in (
    (candidate_comparison, CANDIDATE_OPERATOR_YEAR_PATH), (candidate_total, CANDIDATE_TOTAL_PATH),
    (operator_uplift, CANDIDATE_UPLIFT_OPERATOR_YEAR_PATH), (total_uplift, CANDIDATE_UPLIFT_TOTAL_PATH),
):
    frame.to_csv(path, index=False)

print("Average annual TWh comparison (weather years kept separate in the operator/year file):")
print(total_uplift[total_keys + ["shifted_twh_6h", "shifted_twh_9h", "increase_percent_vs_6h"]].to_string(index=False))

Average annual TWh comparison (weather years kept separate in the operator/year file):
allocation_method utilisation_scenario_id flexibility_scenario_id           shift_mode         model_layer  shifted_twh_6h  shifted_twh_9h  increase_percent_vs_6h
           greedy                 util_80                 flex_20 daily_spatiotemporal        current_only        5.381305        8.175587               51.925721
           greedy                 util_80                 flex_20 daily_spatiotemporal current_plus_future        6.717948       10.216150               52.072476
           greedy                 util_80                 flex_20 same_hour_geographic        current_only        2.200397        3.986821               81.186469
           greedy                 util_80                 flex_20 same_hour_geographic current_plus_future        3.071944        5.414849               76.267833
           greedy                 util_80                 flex_25 daily_spatiotemporal        curr